# Seebeck coefficient from transport data: SPB baseline, family calibration and correction models

**One notebook, end to end.** Starrydata download → S–n dataset → domain rules → formula-compatible families → paper reservation → fold-local calibration → baseline ladder → diagnostics → frozen, documented outputs.

This version replaces the 13 separate notebooks (`seebeck_dataset_starrydata_1`, `seebeck_features_preprocessing_1`, `mstar_calibration_5`, `xgb_ranking_merging_2`, `dimensionless_xgb`, `freeze_setup` and their earlier copies) and implements the corrections from the detailed audit of 24 September 2026.

| | |
|---|---|
| **Runtime** | about 15–25 min on a standard Colab CPU runtime with every section on; about 6 min with `RUN_IMPORTANCE`, `RUN_TRANSFER` and `RUN_SENSITIVITY` off |
| **How to run** | edit Section 1 if needed, then **Runtime → Run all** |
| **Outputs** | `seebeck_pipeline_out/` (and a zip): frozen dataset, config, fold predictions, fitted models, tables, figures, `README.md`, `MANIFEST.json` |
| **Status of results** | development-CV results on papers that were reserved *after* exploratory analysis. Exploratory and methodological, not an independent external validation (see A1) |

### What the audit changed, and where

| Audit item | Change in this notebook | Section |
|---|---|---|
| **A1** reserved papers were used in earlier exploration | They're now called *papers reserved after exploratory analysis*. The same 41 papers are kept fixed, sealed behind a confirmation phrase, and no cell here scores them | 6 |
| **A2** weights ≠ 1 per sample after training filters | Weights are recomputed **after** every training-only filter, inside each fold (`WEIGHTING`). The legacy retained-fraction weighting is available as a sensitivity run | 7, 12 |
| **A3** the "direct / no physics" model still used fitted m* | Renamed *Direct XGBoost on calibrated dimensionless features*. Added a **raw-input XGBoost** (raw n, T, the same transport inputs and a family one-hot, no target-calibrated transform) | 9 |
| **A4** the oracle was presented as a ceiling | Now a **descriptive** residual decomposition with exact definitions, run in-sample and out-of-fold. Ceiling and impossibility claims are removed | 12 |
| **A5** the FeSb2 exclusion removed RuSb2 and FeAs2 too | `FAMILY_EXCLUSION_MODE = "host"` excludes only FeSb2-host compositions. `"family"` restores the old whole-family rule, which must then be justified | 5 |
| **A6** transfer distances were contaminated | m* is recalibrated inside every inner split. Papers that appear in both groups are removed from A's training data before scoring on B | 11 |
| **A7** the pooled vs family comparison changed capacity | Equal capacity by default (`CAPACITY_MODE`). The share of pooled-fallback predictions is reported | 11 |
| **A8** global filters and all-data statistics | Prespecified domain rules are separated from data-adaptive ones. The MAD outlier rule is now fitted **on training papers only** and removes training rows only. The domain is printed next to every headline table | 4, 7 |
| **B1** "sign-changing" meant Hall sign | Labels are now `hall_sign_mixed` and `hall_zero_only`, kept separate. A separate `flag_seebeck_sign_mixed` is QC only | 3 |
| **B2** post-peak rule used only matched rows | Computed on each **original individual S(T) curve** (curve ID kept). Renamed *post-peak screening flag*. A no-filter sensitivity run is included | 3, 12 |
| **B3** auxiliary interpolation extended endpoints | Strict in-range interpolation by default (`AUX_EDGE_TOL_K = 0`). Interpolation is done per curve, with property source, curve count, spread and an endpoint flag kept | 3 |
| **B4** n precedence and pairing | Wording is exact (*sample-level source precedence, nearest-temperature pairing*). A temperature-level fallback option, n-point reuse counts and a narrow-tolerance subset score are added | 3, 12 |
| **B5** first-match family rules | Every matching rule is recorded (`family_candidates`, `family_ambiguous`). Families are called *formula-compatible* | 5 |
| **B6** Hall factor = 1 | Kept as the documented default, plus an acoustic-SPB Hall-factor r_H(η) sensitivity study (masses and CV scores) | 12 |
| **B7** notation | Fixed in the text: log₁₀(μ/[1 cm²/(V s)]), m_rel = m*/mₑ, η relative to the band edge, n/N_c ≈ 0.765 at η = 0, and the loss written as \|\|S\| − S_SPB\| | 2, 7 |
| **B8** wrong-dimension unit alias | `kg*m**2/A/s**3` is removed. Every unit string seen is logged by property (accepted and rejected) | 3 |
| **C1** K-means scaling used all data | Scaling is fitted on training samples only. The K-means section is exploratory and uses development papers only | 8 |
| **C2** seeds were not broad resamples | `FOLD_METHOD = "shuffled_greedy"` orders papers by hash only. Fold stability across seeds is reported, seed SD is labelled *split sensitivity*, and paper-level bootstrap CIs are added | 6, 9 |
| **C3** point-weighted metrics | Every table labels scores as point-weighted and adds macro-sample and macro-paper MAE | 2, 9 |
| **C4** dimensional vs dimensionless on different rows | A **paired** comparison on identical rows, folds, weights and filters | 9 |
| **C5** unreproducible anecdotes | Replaced by a saved train-vs-validation fit diagnostic of the actual models | 12 |
| **C6** reproducibility | Release tag resolved and recorded (or pinned), raw hashes, package versions, per-point OOF predictions with mass provenance, fitted models, a README with the version map, and a SHA-256 manifest of every output | 2, 13 |
| **§5** report numbers | Regenerated from the data into `report_numbers.json` (populations, counts, bounds, mass CVs with fit counts, parameter counts incl. intercept) | 13 |

### Conventions used throughout
* **Target:** |S| (magnitude). The correction target is Δs = s − s_SPB with s = |S|/(k_B/e), k_B/e = 86.17 µV/K.
* **Physics baseline:** single parabolic band (SPB), acoustic-phonon scattering, Hall factor r_H = 1 unless stated. m* is a **calibration constant** fitted per formula-compatible family on training papers only. It is an effective quantity under these approximations, not a band mass.
* **Scores:** *point-weighted* R², MAE and RMSE over pooled out-of-fold points (the headline), plus macro-sample and macro-paper MAE. The ± value is the **SD across fold seeds (split sensitivity)**, not a confidence interval. CIs come from a paper-level bootstrap.
* **Domain:** every score applies only to the eligibility domain printed with it (|S| range, T range, required inputs, excluded hosts).

## 1. Configuration
Every choice that affects results is here. The whole block is saved to `frozen_config.json`. Defaults implement the audit corrections. Where a setting restores legacy behaviour, the comment says so.

In [ ]:
# ============================== DATA SOURCE ==============================
INPUT_MODE = "starrydata"          # "starrydata": rebuild from raw Starrydata files (recommended; required for B2/B3/B8 fixes)
                                   # "matched_points": start from an existing matched_points.csv / seebeck_dataset.zip (legacy fallback)
STARRYDATA_RELEASE = None          # None -> resolve the current release tag via the GitHub API and RECORD it; set a tag string to pin
RAW_DIR = "starrydata_raw"
FORCE_DOWNLOAD = False
RAW_SHA256_EXPECTED = {}           # optional {"curves": "...", "samples": "...", "papers": "..."}; a mismatch stops the run
USE_DRIVE = False                  # True -> mount Google Drive: raw snapshot + outputs are copied to DRIVE_DIR
DRIVE_DIR = "/content/drive/MyDrive/seebeck_pipeline"
OUT_DIR = "seebeck_pipeline_out"

# ============================== DATASET CONSTRUCTION (Section 3) ==============================
TOL_K = 5.0                        # max |T_S - T_n| (K) for nearest-temperature S–n pairing
N_SOURCE_PRECEDENCE = "sample"     # "sample": if a sample has ANY reported n, Hall-derived n is not used for that sample (legacy, documented)
                                   # "temperature": reported n where it covers the S temperature, Hall-derived n elsewhere
AUX_EDGE_TOL_K = 0.0               # 0 = strict interpolation inside each measured curve (no endpoint extension). Legacy: 5.0
POSTPEAK_DROP = 0.90               # post-peak screening flag: T > T(|S|max) of the ORIGINAL curve and |S| < 0.90 |S|max
POSTPEAK_MIN_POINTS = 4            # only curves with >= this many original points are screened
ACCEPT_SUSPECT_UNIT_ALIAS = False  # True re-admits 'kg*m**2/A/s**3' for S (dimension of VOLT, not V/K). Keep False.

# ============================== ELIGIBILITY DOMAIN (Section 4) ==============================
# Prespecified physical / data-quality rules (applied before any split; they DEFINE the evaluation domain)
DROP_FLAGS = ["flag_not_experiment", "flag_not_bulk", "flag_S_range", "flag_n_range", "flag_T_range"]
EXCLUDE_HALL_SIGN_MIXED = True     # samples whose digitized Hall coefficients have both signs
EXCLUDE_HALL_ZERO_ONLY = False     # samples whose only Hall values are exactly 0 (treated as 'unknown' carrier type when kept)
LIMITS = {"T_K": (10, 1300), "log10_n_cm3": (17, 22.5), "S_abs_uVK": (5.0, 1000.0),
          "el_X_mean": (0.7, 4.0), "el_n_valence_mean": (0, 12), "dopant_total_frac": (0, 0.25)}
DEDUP_ACROSS_PAPERS = True         # same composition + T + n + S in a different paper = reused data -> keep the first
T_MIN = 100.0                      # K
REQUIRE_MOBILITY = True            # keep rows with a QC-passed conductivity, so every input set uses identical rows
# Data-adaptive rule (NOT part of the domain): robust-z feature outliers
FEATURE_OUTLIER_MODE = "train_fold"  # "train_fold": median/MAD fitted on each fold's TRAINING rows, removes training rows only
                                     # "global": legacy (fitted on all rows, removes rows before splitting); "off"
ROBUST_Z = 5.0
OUTLIER_FEATURES = ["T_K", "log10_n_cm3", "S_SPB_abs_m1_uVK", "el_X_mean", "el_n_valence_mean"]
# dopant_total_frac is deliberately NOT screened: it is zero-inflated, so its MAD is ~0 or tiny and a robust-z rule would flag
# most doped samples (the legacy global rule skipped it only because its MAD happened to be exactly 0). LIMITS bounds it instead.

# ============================== FAMILIES (Section 5) ==============================
FAMILY_EXCLUSION_MODE = "host"     # "host": exclude only compositions whose host is in EXCLUDE_HOSTS (intended FeSb2 rule)
                                   # "family": exclude every member of EXCLUDE_FAMILIES (legacy; also drops RuSb2, FeAs2 ...); "none"
EXCLUDE_HOSTS = ["FeSb2"]
EXCLUDE_FAMILIES = ["Marcasite MPn2 (FeSb2-type)"]
VERIFIED_TABLE = "verified_hosts.csv"   # optional human-checked (composition, host, family) -> level L1
COMPOSITE_MIN_SHARE = 0.90

# ============================== RESERVATION AND FOLDS (Section 6) ==============================
RESERVED_MODE = "fixed_list"       # "fixed_list": keep the 41 papers reserved on 24 Sep 2026 (continuity; still NOT untouched, see A1)
                                   # "hash": re-derive per-group strata with TEST_SALT (use for NEW, never-inspected papers only)
RESERVED_SIDS_PREVIOUS = [280, 1255, 2020, 2092, 2366, 2461, 2779, 3032, 3112, 3234, 3621, 4003, 4007, 4629, 4748,
                          7747, 7957, 9764, 10904, 12625, 16990, 17021, 21396, 23909, 24621, 27624, 27678, 27870,
                          29002, 29287, 31457, 31473, 31585, 31707, 31838, 31851, 32085, 37312, 37340, 37516, 38440]
TEST_SALT = "seebeck-lockbox-v1"
RESERVED_FRACTION = 0.18
N_FOLDS = 5
SEEDS = [42, 7, 123, 2024, 31415]  # the first one is the primary seed
FOLD_METHOD = "shuffled_greedy"    # papers ordered by SHA-256(seed-SID) only, then greedy balance by points
                                   # "size_first": legacy (largest papers first; many papers keep their fold across seeds)

# ============================== TRAINING POLICY (Section 7) ==============================
TRAIN_EXCLUDE_POSTPEAK = True      # post-peak screened points removed from TRAINING only
WEIGHTING = "equal_retained_sample"  # w = 1 / (retained training points of the sample), recomputed after all training filters
                                     # "retained_fraction": legacy (weights set before filtering)
MIN_TRAIN_PAPERS = 3               # a family needs >= this many training papers for its own m*, else the global m*
M_BOUNDS = (0.05, 20.0)            # m_rel search range
T_REF = 300.0                      # K, tau = T / T_REF

# ============================== MODELS (Sections 9–11) ==============================
CANDIDATE_SETS = {
    "Hall-only | s_SPB input":           ["s_SPB", "ln_n_Nc", "tau"],
    "Hall-only | no s_SPB input":        ["ln_n_Nc", "tau"],
    "Hall + mob + val | s_SPB input":    ["s_SPB", "ln_n_Nc", "tau", "log10_mu", "val_mean"],
    "Hall + mob + val | no s_SPB input": ["ln_n_Nc", "tau", "log10_mu", "val_mean"],
}
PAIRS = [("Hall-only | s_SPB input", "Hall-only | no s_SPB input"),
         ("Hall + mob + val | s_SPB input", "Hall + mob + val | no s_SPB input")]
R2_TOLERANCE, MAE_TOLERANCE_UVK = 0.005, 0.5   # drop s_SPB as an input only if R² falls < 0.005 and MAE rises < 0.5 µV/K
RAW_INPUT_SETS = {                  # A3: raw-input comparison (no target-calibrated mass); family label one-hot is target-free
    "Hall-only":        ["T_K", "log10_n_cm3"],
    "Hall + mob + val": ["T_K", "log10_n_cm3", "log10_mu", "val_mean"],
}
DIMENSIONAL_INPUTS = ["T_K", "log10_n_cm3", "S_SPB_uVK", "el_n_valence_mean", "dopant_total_frac", "log10_sigma_Sm"]  # C4
XGB_PARAMS = dict(n_estimators=400, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8,
                  min_child_weight=5, reg_lambda=1.0, tree_method="hist", random_state=0, n_jobs=-1)
QUAD_RIDGE_ALPHA = 1e-3
N_BOOT = 2000                      # paper-level bootstrap replicates
RUN_KMEANS = True                  # exploratory composition clustering (development papers only)
RUN_IMPORTANCE = True              # drop-column / permutation / SHAP ranking (all seeds x folds)
PERM_REPEATS = 3
RUN_TRANSFER = True                # family transferability + pooling comparison (the slowest section)
TRANSFER_SET = "Hall + mob + val"
CAPACITY_MODE = "equal"            # "equal": every residual model uses XGB_PARAMS | "legacy": small models for single groups
XGB_SMALL = dict(XGB_PARAMS, n_estimators=200, max_depth=3)
MIN_GROUP_PAPERS, MIN_GROUP_POINTS, SELF_FOLDS, MERGE_THRESHOLD = 4, 30, 3, 0.25
RUN_SENSITIVITY = True             # post-peak off, legacy weighting, outlier filter off, Hall factor r_H(eta)

# ============================== RESERVED SUBSET (Section 14) ==============================
EVALUATE_RESERVED = False          # stays False during development
RESERVED_CONFIRM = ""              # must equal CONFIRM_PHRASE below for the one-time evaluation to run
CONFIRM_PHRASE = "I confirm the development protocol is final and will not be changed after this evaluation"

In [ ]:
# Snapshot of every setting (saved to frozen_config.json at the end)
SETTINGS = {k: v for k, v in dict(globals()).items() if k.isupper() and not k.startswith("_")}
print(f"{len(SETTINGS)} settings recorded.")

## 2. Environment and shared library
Imports, version record, output folders, physics, metrics, folds and plotting style. Everything later uses these functions, so each is defined once.

In [ ]:
import sys, os, json, re, time, gzip, hashlib, shutil, platform, warnings, urllib.request, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy.integrate import quad
from scipy.special import expit
from scipy.optimize import minimize_scalar
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
import sklearn
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
try:
    import xgboost as xgb
except ImportError:                      # Colab ships xgboost; elsewhere install it
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "xgboost"], check=True)
    import xgboost as xgb
try:
    from IPython.display import display
except ImportError:
    display = print
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 70)

IN_COLAB = "google.colab" in sys.modules
OUT = Path(OUT_DIR)
for sub in ("tables", "figures", "models", "data"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)

ENV = {"python": platform.python_version(), "platform": platform.platform(), "numpy": np.__version__,
       "pandas": pd.__version__, "scipy": scipy.__version__, "scikit-learn": sklearn.__version__,
       "xgboost": xgb.__version__, "matplotlib": matplotlib.__version__, "colab": IN_COLAB,
       "run_started_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")}
print(json.dumps(ENV, indent=1))
T_START = time.time()

# ---- plotting style (larger labels than the previous figures, per the audit's presentation note)
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
NEUTRAL, INK, INK2, GRID = "#b4b2ac", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({"font.size": 10.5, "axes.titlesize": 11.5, "axes.labelsize": 10.5, "xtick.labelsize": 9.5,
                     "ytick.labelsize": 9.5, "legend.fontsize": 9.5, "axes.edgecolor": INK2, "axes.labelcolor": INK,
                     "xtick.color": INK2, "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False,
                     "figure.dpi": 100, "savefig.dpi": 200, "savefig.bbox": "tight"})

def save_table(df, name, index=True):
    p = OUT / "tables" / f"{name}.csv"; df.to_csv(p, index=index); return p
def save_fig(fig, name):
    fig.savefig(OUT / "figures" / f"{name}.png")
def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()
def sha(x): return hashlib.sha256(x.encode()).hexdigest()
REPORT = {}      # numbers for the written report (saved as report_numbers.json)

### 2.1 Physics: SPB with acoustic-phonon scattering
Unnormalised Fermi integrals $F_j(\eta)=\int_0^\infty x^j/(1+e^{x-\eta})\,dx$. The reduced chemical potential is $\eta=(\mu-E_c)/k_BT$, the finite-temperature chemical potential measured from the band edge of the conducting band (valence-band edge for holes, with the sign convention reversed).

$$n = 4\pi\left(\frac{2 m_{\rm rel} m_e k_B T}{h^2}\right)^{3/2} F_{1/2}(\eta),\qquad |S_{\rm SPB}| = \frac{k_B}{e}\left[\frac{2F_1(\eta)}{F_0(\eta)}-\eta\right]$$

Here $m_{\rm rel}=m^*/m_e$ is the fitted dimensionless ratio, and the SI mass is $m_{\rm rel}\,m_e$. The effective density of states is $N_c = 2(2\pi m_{\rm rel} m_e k_B T/h^2)^{3/2}$, so $n/N_c = (2/\sqrt\pi)F_{1/2}(\eta)$. At $\eta=0$ this gives $n/N_c\approx0.765$ ($\ln(n/N_c)\approx-0.268$). The sign of $\ln(n/N_c)$ is therefore **not** an exact non-degenerate/degenerate boundary: the limits need $n\ll N_c$ or $n\gg N_c$.

**Hall factor.** By default $r_H=1$ ($n = n_H$). The sensitivity study (Section 12) uses the acoustic-SPB value $r_H(\eta)=\tfrac{3}{4}F_{1/2}F_{-1/2}/F_0^2$, which runs from $3\pi/8\approx1.18$ (non-degenerate) to 1 (degenerate), with $n = r_H\,n_H$. Outside the tabulated η range the analytic limits are used.

In [ ]:
KB, H_PL, M_E, E_CH = 1.380649e-23, 6.62607015e-34, 9.1093837015e-31, 1.602176634e-19
KBE_UV = KB / E_CH * 1e6                       # k_B/e = 86.17 µV/K

def _build_fermi_table():
    cache = OUT / "data" / "fermi_table_v2.npz"
    if cache.exists():
        z = np.load(cache); return {k: z[k] for k in z.files}
    etas = np.linspace(-30, 100, 2601)
    F = lambda j, e: quad(lambda x: x ** j * expit(e - x), 0, max(e, 0) + 60, limit=200)[0]
    Fm12 = lambda e: quad(lambda u: 2.0 * expit(e - u * u), 0, np.sqrt(max(e, 0) + 60), limit=200)[0]   # x = u²
    t = {"eta": etas, "F12": np.array([F(0.5, e) for e in etas]), "F1": np.array([F(1.0, e) for e in etas]),
         "F0": np.logaddexp(0, etas), "Fm12": np.array([Fm12(e) for e in etas])}
    t["rH"] = 0.75 * t["F12"] * t["Fm12"] / t["F0"] ** 2
    np.savez(cache, **t); return t

t0 = time.time(); FT = _build_fermi_table()
_LOGF12 = np.log(FT["F12"]); _LOGG = np.log(FT["F12"] / FT["rH"])
print(f"Fermi table ready ({time.time() - t0:.1f} s); r_H range {FT['rH'].min():.4f}–{FT['rH'].max():.4f} (limits 1 and 3π/8 = {3*np.pi/8:.4f})")

def spb_eta(n_m3, T_K, m_rel=1.0, hall_factor=False):
    """Reduced chemical potential eta from n (m^-3), T (K), m_rel.
    hall_factor=True treats n as the Hall density n_H and solves n = r_H(eta) n_H self-consistently."""
    n_m3, T_K, m_rel = (np.asarray(v, float) for v in (n_m3, T_K, m_rel))
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        q = n_m3 / (4 * np.pi * (2 * m_rel * M_E * KB * T_K / H_PL ** 2) ** 1.5)   # F12 (or F12/rH)
        table = _LOGG if hall_factor else _LOGF12
        eta = np.interp(np.log(q), table, FT["eta"])
        lo, hi = np.log(q) < table[0], np.log(q) > table[-1]
        q_lo = q * (3 * np.pi / 8) if hall_factor else q                          # non-degenerate: r_H -> 3π/8
        eta = np.where(lo, np.log(q_lo * 2 / np.sqrt(np.pi)), eta)
        e_hi = (1.5 * q) ** (2 / 3); e_hi = (1.5 * q / (1 + np.pi ** 2 / (8 * e_hi ** 2))) ** (2 / 3)
        eta = np.where(hi, e_hi, eta)
    return eta, lo, hi

def spb_seebeck_abs_uVK(n_m3, T_K, m_rel=1.0, hall_factor=False):
    """|S_SPB| in µV/K (acoustic-phonon scattering). Analytic limits outside -30 <= eta <= 100."""
    eta, lo, hi = spb_eta(n_m3, T_K, m_rel, hall_factor)
    with np.errstate(divide="ignore", invalid="ignore"):
        S = np.where(lo, 2 - eta, np.where(hi, np.pi ** 2 / (3 * eta),
                     2 * np.interp(eta, FT["eta"], FT["F1"]) / np.interp(eta, FT["eta"], FT["F0"]) - eta))
    return S * KBE_UV

def N_c(T_K, m_rel):
    """Effective density of states (m^-3): 2 (2π m_rel m_e k_B T / h²)^{3/2}."""
    return 2 * (2 * np.pi * np.asarray(m_rel, float) * M_E * KB * np.asarray(T_K, float) / H_PL ** 2) ** 1.5

# self-checks against May & Snyder, Table 1 (eta = 0, T = 300 K, m_rel = 1): n ≈ 1.920e19 cm^-3, |S| ≈ 204.5 µV/K
_n0 = 4 * np.pi * (2 * M_E * KB * 300 / H_PL ** 2) ** 1.5 * np.interp(0, FT["eta"], FT["F12"])
assert abs(_n0 * 1e-6 / 1.9201e19 - 1) < 1e-3 and abs(spb_seebeck_abs_uVK(_n0, 300.0) - 204.50) < 0.05
assert abs(np.interp(0, FT["eta"], 2 / np.sqrt(np.pi) * FT["F12"]) - 0.765) < 1e-3
print(f"check: eta=0, 300 K, m_rel=1 -> n = {_n0*1e-6:.4e} cm^-3, |S| = {float(spb_seebeck_abs_uVK(_n0, 300.0)):.2f} µV/K  ✓")

### 2.2 Effective-mass calibration
For a set of training rows, $m_{\rm rel}$ minimises the weighted mean $\big|\,|S| - S_{\rm SPB}(n,T;m_{\rm rel})\big|$ (bounded search in $\ln m_{\rm rel}$). Each family with ≥ `MIN_TRAIN_PAPERS` training papers gets its own value. Every other row uses one global value fitted on all training rows. The fitted family masses are **supervised parameters** (they use training labels), so they are always refitted on the training papers of each split.

In [ ]:
def fit_mstar(S_abs, n_m3, T_K, w, hall_factor=False):
    S_abs, n_m3, T_K, w = (np.asarray(v, float) for v in (S_abs, n_m3, T_K, w))
    loss = lambda lm: np.sum(w * np.abs(S_abs - spb_seebeck_abs_uVK(n_m3, T_K, np.exp(lm), hall_factor))) / w.sum()
    return float(np.exp(minimize_scalar(loss, bounds=np.log(M_BOUNDS), method="bounded", options={"xatol": 1e-4}).x))

def fit_mass_table(train, hall_factor=False):
    """-> ({family: m_rel}, global m_rel). `train` needs S_abs_uVK, n_m3, T_K, w, family, SID."""
    glob = fit_mstar(train.S_abs_uVK, train.n_m3, train.T_K, train.w, hall_factor)
    fam = {f: fit_mstar(g.S_abs_uVK, g.n_m3, g.T_K, g.w, hall_factor)
           for f, g in train.groupby("family") if g["SID"].nunique() >= MIN_TRAIN_PAPERS}
    return fam, glob

def apply_masses(df, fam, glob):
    m = df["family"].map(fam)
    return m.fillna(glob).to_numpy(float), np.where(m.notna(), "family", "global")

### 2.3 Metrics, bootstrap and folds
* **Point-weighted** (headline): R² = 1 − SSE/SST, MAE and RMSE over all pooled out-of-fold points. Dense curves and papers with many samples count more.
* **Macro-sample MAE:** mean over samples of each sample's MAE. **Macro-paper MAE:** mean over papers of each paper's MAE. These matter when the unit of deployment is a new sample or a new paper.
* **Paper-level bootstrap:** papers are resampled with replacement (cluster bootstrap) on the primary seed's out-of-fold predictions. It gives 95 % CIs, and paired CIs for differences between models.
* **Folds:** papers are ordered by SHA-256(`seed-SID`) and assigned greedily to the fold with the fewest points so far. Each paper is in exactly one fold, and the assignment is library-independent. With `FOLD_METHOD="size_first"` (legacy), large papers go first, so many papers keep the same fold label across seeds.

In [ ]:
def r2(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float); return float(1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2))
def mae(y, p): return float(np.mean(np.abs(np.asarray(y, float) - np.asarray(p, float))))
def rmse(y, p): return float(np.sqrt(np.mean((np.asarray(y, float) - np.asarray(p, float)) ** 2)))

def score_s(y_s, p_s, frame):
    """Scores for predictions in s units; MAE/RMSE reported in µV/K."""
    e = np.abs(np.asarray(y_s) - np.asarray(p_s)) * KBE_UV
    return {"R2": r2(y_s, p_s), "MAE_uVK": float(e.mean()), "RMSE_uVK": rmse(y_s, p_s) * KBE_UV,
            "MAE_macro_sample_uVK": float(pd.Series(e).groupby(frame["sample_key"].to_numpy()).mean().mean()),
            "MAE_macro_paper_uVK": float(pd.Series(e).groupby(frame["SID"].to_numpy()).mean().mean())}

class PaperBootstrap:
    """Cluster bootstrap over papers for pooled R² and MAE (vectorised with per-paper sufficient statistics)."""
    def __init__(self, sid, n_boot=N_BOOT, seed=0):
        codes, self.papers = pd.factorize(np.asarray(sid)); self.codes = codes; P = len(self.papers)
        rng = np.random.default_rng(seed)
        self.C = np.stack([np.bincount(rng.integers(0, P, P), minlength=P) for _ in range(n_boot)]).astype(float)
        self.npts = np.bincount(codes, minlength=P).astype(float)
    def _stats(self, y, p):
        s = lambda v: np.bincount(self.codes, weights=v, minlength=len(self.papers))
        return s(np.abs(y - p)), s((y - p) ** 2), s(y), s(y ** 2)
    def metrics(self, y, p, scale=KBE_UV):
        y, p = np.asarray(y, float), np.asarray(p, float); a, se, sy, sy2 = self._stats(y, p)
        N = self.C @ self.npts
        mae_b = (self.C @ a) / N * scale
        sst = self.C @ sy2 - (self.C @ sy) ** 2 / N
        r2_b = 1 - (self.C @ se) / sst
        return mae_b, r2_b
    def ci(self, y, p):
        m, r = self.metrics(y, p)
        return {"MAE_CI95": tuple(np.round(np.percentile(m, [2.5, 97.5]), 2)), "R2_CI95": tuple(np.round(np.percentile(r, [2.5, 97.5]), 4))}
    def paired(self, y, p_a, p_b):
        ma, ra = self.metrics(y, p_a); mb, rb = self.metrics(y, p_b)
        dm, dr = mb - ma, rb - ra
        return {"dMAE_uVK (b-a)": float(np.mean(dm)), "dMAE_CI95": tuple(np.round(np.percentile(dm, [2.5, 97.5]), 2)),
                "dR2 (b-a)": float(np.mean(dr)), "dR2_CI95": tuple(np.round(np.percentile(dr, [2.5, 97.5]), 4)),
                "P(b better MAE)": float(np.mean(dm < 0))}

def make_folds(df, n_folds, seed, method=FOLD_METHOD):
    pts = df.groupby("SID").size()
    key = {sid: sha(f"{seed}-{sid}") for sid in pts.index}
    order = (sorted(pts.index, key=lambda s: (-pts[s], key[s])) if method == "size_first"
             else sorted(pts.index, key=lambda s: key[s]))
    load, fold_of = [0] * n_folds, {}
    for sid in order:
        k = min(range(n_folds), key=lambda i: (load[i], i)); fold_of[sid] = k; load[k] += pts[sid]
    return df["SID"].map(fold_of).to_numpy()

def splits_from(col):
    col = np.asarray(col); return [(np.where(col != k)[0], np.where(col == k)[0]) for k in sorted(np.unique(col))]

## 3. Data acquisition and S–n dataset construction
### 3.1 Download (release resolved, recorded and hashed)
The earlier downloader used `/releases/latest/download/`, so the exact data release could not be reconstructed. Here the release **tag** is resolved through the GitHub API (or pinned with `STARRYDATA_RELEASE`). The tag, asset URLs, file sizes and SHA-256 hashes are recorded. With `USE_DRIVE`, a raw snapshot is copied to Drive so the run can be repeated without the network.

In [ ]:
PROJECT = "ThermoelectricMaterials"
REPO = "starrydata/starrydata_datasets"
PROVENANCE = {"input_mode": INPUT_MODE}

def resolve_release(tag):
    info = {"requested": tag}
    if tag:
        info.update(tag=tag, base_url=f"https://github.com/{REPO}/releases/download/{tag}/")
        return info
    try:
        req = urllib.request.Request(f"https://api.github.com/repos/{REPO}/releases/latest",
                                     headers={"Accept": "application/vnd.github+json", "User-Agent": "seebeck-pipeline"})
        with urllib.request.urlopen(req, timeout=30) as r:
            j = json.load(r)
        info.update(tag=j["tag_name"], published_at=j.get("published_at"),
                    base_url=f"https://github.com/{REPO}/releases/download/{j['tag_name']}/")
    except Exception as e:                           # offline / rate-limited: fall back, but say so loudly
        info.update(tag="UNRESOLVED", base_url=f"https://github.com/{REPO}/releases/latest/download/",
                    warning=f"release tag could not be resolved ({type(e).__name__}); set STARRYDATA_RELEASE to pin")
    return info

def fetch_raw(raw_dir, release, force):
    raw_dir = Path(raw_dir); raw_dir.mkdir(parents=True, exist_ok=True); files = {}
    for kind in ("papers", "samples", "curves"):
        fn = f"{PROJECT}_{kind}.csv.gz"; p = raw_dir / fn
        drive_copy = Path(DRIVE_DIR) / "raw" / release.get("tag", "UNRESOLVED") / fn
        if not p.exists() and USE_DRIVE and drive_copy.exists() and not force:
            shutil.copy(drive_copy, p); print(f"restored {fn} from Drive snapshot")
        if force or not p.exists():
            print(f"downloading {fn} ({release['tag']}) ..."); urllib.request.urlretrieve(release["base_url"] + fn, p)
        files[kind] = {"file": fn, "url": release["base_url"] + fn, "bytes": p.stat().st_size, "sha256": sha256_file(p)}
        if RAW_SHA256_EXPECTED.get(kind) and RAW_SHA256_EXPECTED[kind] != files[kind]["sha256"]:
            raise ValueError(f"{fn}: SHA-256 differs from RAW_SHA256_EXPECTED; the source data changed")
        if USE_DRIVE and not drive_copy.exists():
            drive_copy.parent.mkdir(parents=True, exist_ok=True); shutil.copy(p, drive_copy)
    return files

if INPUT_MODE == "starrydata":
    RELEASE = resolve_release(STARRYDATA_RELEASE)
    PROVENANCE["release"] = RELEASE
    PROVENANCE["raw_files"] = fetch_raw(RAW_DIR, RELEASE, FORCE_DOWNLOAD)
    if RELEASE["tag"] == "UNRESOLVED":
        print("WARNING:", RELEASE["warning"])
    print(json.dumps(PROVENANCE, indent=1))

### 3.2 Properties, units and composition helpers
* **Units.** Only dimensionally correct aliases are accepted. `kg*m**2/A/s**3` (the SI form of the **volt**, missing K⁻¹) is no longer accepted for S (audit B8). Every unit string seen for S, n and R_H is counted by property, both accepted and rejected (`unit_log.csv`), so any rejected curves can be inspected rather than silently lost.
* **Composition descriptors.** Mean valence electrons come from one explicit table (s/p block: group, or group − 10 for groups ≥ 13; d block: group number; f block: common valence in compounds). Pauling electronegativities come from an explicit table identical to the pymatgen values used before, so pymatgen is no longer a dependency. If more than 10 % of the atoms lack a value, the result is NaN.

In [ ]:
PROP_ALIASES = {"Seebeck coefficient": "S", "thermopower": "S", "Thermopower": "S", "S": "S",
                "Carrier concentration": "n", "Hall coefficient": "RH"}
UNITS = {"S": {"V*K^(-1)": 1.0, "V/K": 1.0, "V*K**(-1)": 1.0, "kg*m**2/(A*K*s**3)": 1.0, "kg*m**2/A/K/s**3": 1.0},
         "n": {"m^(-3)": 1.0, "m**(-3)": 1.0, "1/m**3": 1.0},
         "RH": {"m^3*C^(-1)": 1.0, "m**3/C": 1.0, "m**3/(A*s)": 1.0}}
SUSPECT_UNITS = {"S": {"kg*m**2/A/s**3": 1.0}}      # dimension of volt; accepted only if ACCEPT_SUSPECT_UNIT_ALIAS
if ACCEPT_SUSPECT_UNIT_ALIAS:
    UNITS["S"].update(SUSPECT_UNITS["S"])

def x_to_kelvin(prop_x, unit_x, x):
    if prop_x in ("Temperature", "T") and unit_x == "K": return x
    if prop_x == "Inverse temperature" and unit_x == "K^(-1)":
        with np.errstate(divide="ignore"): return 1.0 / x
    return None

_TOKEN = re.compile(r"([A-Z][a-z]?)(\d*\.?\d*)|(\()|(\))(\d*\.?\d*)")
def parse_formula(formula):
    """'Bi0.5Sb1.5Te3', '(Bi2Te3)0.9(Sb2Te3)0.1' -> normalised atomic fractions, or None if unparseable."""
    if not isinstance(formula, str) or not formula.strip(): return None
    s = re.sub(r"[\s\-·•]", "", formula); stack = [{}]; pos = 0
    for m_ in _TOKEN.finditer(s):
        if m_.start() != pos: return None
        pos = m_.end(); el, num, lpar, rpar, rnum = m_.groups()
        if el: stack[-1][el] = stack[-1].get(el, 0) + (float(num) if num else 1.0)
        elif lpar: stack.append({})
        elif rpar:
            if len(stack) == 1: return None
            grp = stack.pop(); k = float(rnum) if rnum else 1.0
            for e, v in grp.items(): stack[-1][e] = stack[-1].get(e, 0) + v * k
    if pos != len(s) or len(stack) != 1 or not stack[0]: return None
    tot = sum(stack[0].values())
    return {e: v / tot for e, v in stack[0].items() if v > 0} if tot > 0 else None

VALENCE = {"H":1,"Li":1,"Be":2,"B":3,"C":4,"N":5,"O":6,"F":7,"Na":1,"Mg":2,"Al":3,"Si":4,"P":5,"S":6,"Cl":7,
 "K":1,"Ca":2,"Sc":3,"Ti":4,"V":5,"Cr":6,"Mn":7,"Fe":8,"Co":9,"Ni":10,"Cu":11,"Zn":12,"Ga":3,"Ge":4,"As":5,"Se":6,"Br":7,
 "Rb":1,"Sr":2,"Y":3,"Zr":4,"Nb":5,"Mo":6,"Tc":7,"Ru":8,"Rh":9,"Pd":10,"Ag":11,"Cd":12,"In":3,"Sn":4,"Sb":5,"Te":6,"I":7,
 "Cs":1,"Ba":2,"Hf":4,"Ta":5,"W":6,"Re":7,"Os":8,"Ir":9,"Pt":10,"Au":11,"Hg":12,"Tl":3,"Pb":4,"Bi":5,"Po":6,"At":7,
 "La":3,"Ce":3,"Pr":3,"Nd":3,"Pm":3,"Sm":3,"Eu":2,"Gd":3,"Tb":3,"Dy":3,"Ho":3,"Er":3,"Tm":3,"Yb":2,"Lu":3,
 "Ac":3,"Th":4,"Pa":5,"U":6,"Np":5,"Pu":4}
F_BLOCK = {"La","Ce","Pr","Nd","Pm","Sm","Eu","Gd","Tb","Dy","Ho","Er","Tm","Yb","Lu","Ac","Th","Pa","U","Np","Pu"}
PAULING_X = {"H":2.2,"Li":0.98,"Be":1.57,"B":2.04,"C":2.55,"N":3.04,"O":3.44,"F":3.98,"Na":0.93,"Mg":1.31,"Al":1.61,
 "Si":1.9,"P":2.19,"S":2.58,"Cl":3.16,"K":0.82,"Ca":1.0,"Sc":1.36,"Ti":1.54,"V":1.63,"Cr":1.66,"Mn":1.55,"Fe":1.83,
 "Co":1.88,"Ni":1.91,"Cu":1.9,"Zn":1.65,"Ga":1.81,"Ge":2.01,"As":2.18,"Se":2.55,"Br":2.96,"Rb":0.82,"Sr":0.95,"Y":1.22,
 "Zr":1.33,"Nb":1.6,"Mo":2.16,"Tc":1.9,"Ru":2.2,"Rh":2.28,"Pd":2.2,"Ag":1.93,"Cd":1.69,"In":1.78,"Sn":1.96,"Sb":2.05,
 "Te":2.1,"I":2.66,"Cs":0.79,"Ba":0.89,"La":1.1,"Ce":1.12,"Pr":1.13,"Nd":1.14,"Pm":1.13,"Sm":1.17,"Eu":1.2,"Gd":1.2,
 "Tb":1.1,"Dy":1.22,"Ho":1.23,"Er":1.24,"Tm":1.25,"Yb":1.1,"Lu":1.27,"Hf":1.3,"Ta":1.5,"W":2.36,"Re":1.9,"Os":2.2,
 "Ir":2.2,"Pt":2.28,"Au":2.54,"Hg":2.0,"Tl":1.62,"Pb":2.33,"Bi":2.02,"Po":2.0,"At":2.2,"Ac":1.1,"Th":1.3,"Pa":1.5,
 "U":1.38,"Np":1.36,"Pu":1.28}
MAIN_FRAC = 0.05                   # elements below 5 at.% count as dopants

def composition_descriptors(frac):
    """-> dict of target-free descriptors, or status if the formula has unknown symbols."""
    if frac is None: return {"composition_status": "unparsed_or_missing"}
    if any(e not in VALENCE for e in frac): return {"composition_status": "unknown_element_symbol"}
    out = {"composition_status": "ok", "n_elements": len(frac),
           "n_dopants": sum(v < MAIN_FRAC for v in frac.values()),
           "dopant_total_frac": sum(v for v in frac.values() if v < MAIN_FRAC),
           "el_n_valence_mean": sum(v * VALENCE[e] for e, v in frac.items()),
           "has_f_block": bool(F_BLOCK & set(frac))}
    xs = {e: v for e, v in frac.items() if e in PAULING_X}
    wsum = sum(xs.values())
    out["el_X_mean"] = sum(v * PAULING_X[e] for e, v in xs.items()) / wsum if wsum >= 0.9 else np.nan
    return out

### 3.3 Build the matched S–n table from the raw curves
What changed compared with the earlier dataset notebook:

| Item | Now |
|---|---|
| **Carrier-type label (B1)** | From the **Hall coefficient sign** only: `n` (all R_H < 0), `p` (all R_H > 0), **`hall_sign_mixed`** (both signs present), **`hall_zero_only`** (only R_H = 0), `unknown` (no Hall curve). Positive R_H stays `low_check_source` (a digitised magnitude can look like p-type). |
| **Seebeck sign (B1)** | `flag_seebeck_sign_mixed` marks samples whose original S points include both signs. It's a QC flag only, never used for selection or as a label. |
| **Carrier density (B4)** | *Sample-level source precedence* (`N_SOURCE_PRECEDENCE="sample"`, legacy) or *temperature-level fallback*. Pairing is *nearest-temperature* within `TOL_K`, and one n point may pair with several S points (reuse counted in `n_point_reuse`). Reported concentrations may themselves be Hall-derived or model-derived in the source papers. That has to be checked in the papers and is not assumed here. |
| **Post-peak screening flag (B2)** | Computed on each **original individual S(T) curve** (`S_curve_uid` = the Starrydata curve row), including S points that never found an n partner. It is a heuristic screen. A drop in \|S\| after a maximum is *compatible with* bipolar conduction but does not prove it. |
| **Auxiliary σ, μ, κ (B3)** | Each curve is interpolated separately, **strictly inside its measured T range** (`AUX_EDGE_TOL_K = 0`), log-linear for σ and μ. When several curves cover a temperature, the median is used and the curve count, log-spread, source property names and an endpoint-extension flag are kept. |

In [ ]:
def explode_curves(curves, log):
    c = curves.assign(prop=curves["prop_y"].map(PROP_ALIASES))
    c = c[c["prop"].notna()]
    rows, dropped, units = [], {"bad_unit_y": 0, "bad_x_axis": 0, "bad_json": 0, "len_mismatch": 0}, {}
    for r in c.itertuples():
        mult = UNITS[r.prop].get(r.unit_y)
        ukey = (r.prop, r.unit_y, mult is not None, r.unit_y in SUSPECT_UNITS.get(r.prop, {}))
        units[ukey] = units.get(ukey, 0) + 1
        if mult is None: dropped["bad_unit_y"] += 1; continue
        try:
            x = np.asarray(json.loads(r.x), dtype=float); y = np.asarray(json.loads(r.y), dtype=float) * mult
        except (TypeError, ValueError):
            dropped["bad_json"] += 1; continue
        if x.shape != y.shape: dropped["len_mismatch"] += 1; continue
        T = x_to_kelvin(r.prop_x, r.unit_x, x)
        if T is None: dropped["bad_x_axis"] += 1; continue
        ok = np.isfinite(T) & np.isfinite(y)
        rows.append(pd.DataFrame({"SID": r.SID, "sample_id": r.sample_id, "figure_id": r.figure_id,
                                  "figure_name": r.figure_name, "curve_uid": r.Index, "prop": r.prop,
                                  "T_K": T[ok], "value": y[ok]}))
    log["curves_dropped"] = dropped
    log["unit_log"] = [{"prop": k[0], "unit_y": k[1], "accepted": k[2], "suspect_alias": k[3], "curves": v} for k, v in units.items()]
    pts = pd.concat(rows, ignore_index=True); before = len(pts)
    key = pts.assign(_T=pts["T_K"].round(1), _v=pts["value"].round(12))
    pts = pts[~key.duplicated(["SID", "sample_id", "prop", "_T", "_v"])].reset_index(drop=True)
    log["duplicate_points_removed"] = before - len(pts)
    log["points_by_prop"] = pts["prop"].value_counts().to_dict()
    return pts

def hall_labels(pts):
    rh = pts[pts["prop"] == "RH"].groupby(["SID", "sample_id"])["value"]
    out = pd.DataFrame({"n_neg": rh.apply(lambda v: int((v < 0).sum())), "n_pos": rh.apply(lambda v: int((v > 0).sum())),
                        "n_zero": rh.apply(lambda v: int((v == 0).sum()))}).reset_index()
    out["carrier_type"] = np.select([(out.n_neg > 0) & (out.n_pos == 0), (out.n_pos > 0) & (out.n_neg == 0),
                                     (out.n_pos > 0) & (out.n_neg > 0)], ["n", "p", "hall_sign_mixed"], default="hall_zero_only")
    out["carrier_type_confidence"] = np.select([out.carrier_type == "n", out.carrier_type == "p"], ["high", "low_check_source"], "n/a")
    return out[["SID", "sample_id", "carrier_type", "carrier_type_confidence"]]

def s_points_with_screen(pts):
    """Seebeck points with the post-peak screening flag computed on each ORIGINAL curve."""
    S = pts[pts["prop"] == "S"].rename(columns={"value": "S_VK", "figure_id": "S_figure_id", "curve_uid": "S_curve_uid"}).copy()
    S["S_abs"] = S["S_VK"].abs()
    g = S.groupby("S_curve_uid")
    S["curve_points_original"] = g["T_K"].transform("size")
    imax = g["S_abs"].idxmax()
    peak = S.loc[imax, ["S_curve_uid", "T_K", "S_abs"]].rename(columns={"T_K": "T_at_Smax", "S_abs": "Smax"})
    S = S.merge(peak, on="S_curve_uid", how="left")
    S["flag_postpeak_screen"] = ((S.curve_points_original >= POSTPEAK_MIN_POINTS) & (S.T_K > S.T_at_Smax)
                                 & (S.S_abs < POSTPEAK_DROP * S.Smax))
    sgn = S[S.S_abs > 0].groupby(["SID", "sample_id"])["S_VK"].agg(lambda v: bool((v > 0).any() and (v < 0).any()))
    S = S.merge(sgn.rename("flag_seebeck_sign_mixed").reset_index(), on=["SID", "sample_id"], how="left")
    S["flag_seebeck_sign_mixed"] = S["flag_seebeck_sign_mixed"].fillna(False).astype(bool)
    return S.drop(columns=["S_abs"])

def n_candidates(pts):
    conc = pts[pts["prop"] == "n"].assign(n_m3=lambda d: d["value"].abs(), n_source="reported_n")
    hall = pts[(pts["prop"] == "RH") & (pts["value"] != 0)].assign(n_m3=lambda d: 1.0 / (E_CH * d["value"].abs()),
                                                                   n_source="hall_1/(e|RH|)")
    n = pd.concat([conc, hall], ignore_index=True)
    n["n_reported_sign"] = np.sign(n["value"])
    n = n.rename(columns={"figure_id": "n_figure_id", "T_K": "T_n_K", "curve_uid": "n_curve_uid"})
    n["n_uid"] = np.arange(len(n))
    return n[["SID", "sample_id", "n_figure_id", "n_curve_uid", "n_uid", "T_n_K", "n_m3", "n_source", "n_reported_sign"]]

def pair_nearest(S, n, tol):
    S = S.assign(key=S["SID"].astype(str) + "_" + S["sample_id"].astype(str))
    n = n.assign(key=n["SID"].astype(str) + "_" + n["sample_id"].astype(str))
    S = S[S["key"].isin(set(n["key"]))].sort_values("T_K")
    m = pd.merge_asof(S, n.drop(columns=["SID", "sample_id"]).sort_values("T_n_K"), left_on="T_K", right_on="T_n_K",
                      by="key", direction="nearest", tolerance=tol)
    return m[m["n_m3"].notna()].drop(columns="key")

def match_s_to_n(S, n, tol, precedence):
    rep, hall = n[n.n_source == "reported_n"], n[n.n_source != "reported_n"]
    if precedence == "sample":
        have = set(zip(rep.SID, rep.sample_id))
        hall = hall[[k not in have for k in zip(hall.SID, hall.sample_id)]]
        m = pair_nearest(S, pd.concat([rep, hall]), tol)
    else:                                               # temperature-level fallback
        m1 = pair_nearest(S, rep, tol)
        rest = S[~S["s_uid"].isin(m1["s_uid"])]
        m = pd.concat([m1, pair_nearest(rest, hall, tol)], ignore_index=True)
    m["dT_K"] = (m["T_K"] - m["T_n_K"]).abs()
    m["n_point_reuse"] = m.groupby("n_uid")["n_uid"].transform("size")
    return m

AUX_PROPS = {  # prop_y -> (canonical, {unit: converter}, log-space?)   leak_* are computed FROM S: QC only, never inputs
    "Electrical conductivity":  ("sigma_Sm", {"ohm^(-1)*m^(-1)": lambda v: v, "S/m": lambda v: v}, True),
    "Electrical resistivity":   ("sigma_Sm", {"ohm*m": lambda v: 1.0 / v}, True),
    "Carrier mobility":         ("mu_m2Vs", {"m^2*V^(-1)*s^(-1)": lambda v: v, "A*s**2/kg": lambda v: v}, True),
    "Hall mobility":            ("mu_m2Vs", {"m^2*V^(-1)*s^(-1)": lambda v: v, "A*s**2/kg": lambda v: v}, True),
    "Mobility":                 ("mu_m2Vs", {"m^2*V^(-1)*s^(-1)": lambda v: v, "A*s**2/kg": lambda v: v}, True),
    "Thermal conductivity":     ("kappa_WmK", {"W*m^(-1)*K^(-1)": lambda v: v}, False),
    "total thermal conductivity": ("kappa_WmK", {"W*m^(-1)*K^(-1)": lambda v: v}, False),
    "Power factor":             ("leak_PF_WmK2", {"W*m^(-1)*K^(-2)": lambda v: v}, False),
    "ZT":                       ("leak_ZT", {"-": lambda v: v}, False),
}

def explode_aux(curves, keys):
    c = curves[curves["prop_y"].isin(AUX_PROPS)]
    c = c[[k in keys for k in zip(c["SID"], c["sample_id"])]]
    rows = []
    for r in c.itertuples():
        canon, units, _ = AUX_PROPS[r.prop_y]; conv = units.get(r.unit_y)
        if conv is None: continue
        try:
            x = np.asarray(json.loads(r.x), dtype=float); y = np.asarray(json.loads(r.y), dtype=float)
        except (TypeError, ValueError):
            continue
        T = x_to_kelvin(r.prop_x, r.unit_x, x) if x.shape == y.shape else None
        if T is None: continue
        with np.errstate(divide="ignore", invalid="ignore"): v = conv(y)
        ok = np.isfinite(T) & np.isfinite(v)
        rows.append(pd.DataFrame({"SID": r.SID, "sample_id": r.sample_id, "curve_uid": r.Index, "source_prop": r.prop_y,
                                  "prop": canon, "T_K": T[ok], "value": v[ok]}))
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(
        columns=["SID", "sample_id", "curve_uid", "source_prop", "prop", "T_K", "value"])

def attach_aux(m, aux, edge_tol):
    """Per-curve interpolation strictly inside each curve's T range (+edge_tol if > 0), combined by median."""
    logspace = {canon: lg for canon, _, lg in AUX_PROPS.values()}
    pos_of = m.groupby(["SID", "sample_id"]).indices
    Tq_all = m["T_K"].to_numpy(float)
    m = m.copy()
    for canon in sorted(aux["prop"].unique()):
        a = aux[aux["prop"] == canon]
        if logspace[canon]: a = a[a["value"] > 0]
        recs = []
        for (sid, smp, cu, src), g in a.groupby(["SID", "sample_id", "curve_uid", "source_prop"]):
            pos = pos_of.get((sid, smp))
            if pos is None: continue
            g = g.groupby("T_K", as_index=False)["value"].mean().sort_values("T_K")
            Tc = g["T_K"].to_numpy(); yc = np.log(g["value"].to_numpy()) if logspace[canon] else g["value"].to_numpy()
            Tq = Tq_all[pos]
            inside = (Tq >= Tc.min()) & (Tq <= Tc.max())
            ext = ~inside & (Tq >= Tc.min() - edge_tol) & (Tq <= Tc.max() + edge_tol) if edge_tol > 0 else np.zeros_like(inside)
            keep = inside | ext
            if not keep.any(): continue
            recs.append(pd.DataFrame({"pos": pos[keep], "v": np.interp(Tq[keep], Tc, yc), "ext": ext[keep], "src": src}))
        if not recs:
            m[canon] = np.nan; continue
        R = pd.concat(recs).groupby("pos").agg(v=("v", "median"), n_curves=("v", "size"), vmin=("v", "min"), vmax=("v", "max"),
                                                ext=("ext", "any"), src=("src", lambda s: "|".join(sorted(set(s)))))
        val = np.exp(R["v"]) if logspace[canon] else R["v"]
        m[canon] = np.nan; m.loc[R.index, canon] = val.to_numpy()
        if not canon.startswith("leak_"):
            m[f"{canon}_n_curves"] = 0; m.loc[R.index, f"{canon}_n_curves"] = R["n_curves"].to_numpy()
            m[f"{canon}_spread"] = np.nan
            m.loc[R.index, f"{canon}_spread"] = ((R.vmax - R.vmin) / (np.log(10) if logspace[canon] else 1)).to_numpy()
            m[f"{canon}_endpoint_ext"] = False; m.loc[R.index, f"{canon}_endpoint_ext"] = R["ext"].to_numpy()
            m[f"{canon}_source"] = ""; m.loc[R.index, f"{canon}_source"] = R["src"].to_numpy()
    return m

def parse_sample_info(s):
    try: d = json.loads(s) if isinstance(s, str) else {}
    except ValueError: d = {}
    get = lambda k: ((d.get(k) or {}).get("category") or "").strip() or None
    return {"data_type": get("DataType"), "form": get("Form"), "material_family_meta": get("MaterialFamily"),
            "fabrication": get("FabricationProcess")}

In [ ]:
DLOG = {}
if INPUT_MODE == "starrydata":
    t0 = time.time()
    raw = {k: Path(RAW_DIR) / v["file"] for k, v in PROVENANCE["raw_files"].items()}
    curves = pd.read_csv(raw["curves"], low_memory=False)
    samples = pd.read_csv(raw["samples"], low_memory=False)
    papers = pd.read_csv(raw["papers"], low_memory=False, usecols=["SID", "DOI"])
    DLOG["raw_rows"] = {"curves": len(curves), "samples": len(samples), "papers": len(papers)}
    pts = explode_curves(curves, DLOG)
    S_pts = s_points_with_screen(pts); S_pts["s_uid"] = np.arange(len(S_pts))
    ctype = hall_labels(pts)
    ncand = n_candidates(pts)
    m = match_s_to_n(S_pts, ncand, TOL_K, N_SOURCE_PRECEDENCE)
    meta = pd.DataFrame([parse_sample_info(v) for v in samples["sample_info"]])
    smp = pd.concat([samples[["SID", "sample_id", "sample_name", "composition"]], meta], axis=1).drop_duplicates(["SID", "sample_id"])
    m = (m.merge(smp, on=["SID", "sample_id"], how="left").merge(ctype, on=["SID", "sample_id"], how="left")
          .merge(papers.drop_duplicates("SID"), on="SID", how="left"))
    m["carrier_type"] = m["carrier_type"].fillna("unknown"); m["carrier_type_confidence"] = m["carrier_type_confidence"].fillna("n/a")
    keys = set(zip(m["SID"], m["sample_id"]))
    aux = explode_aux(curves, keys)
    m = attach_aux(m.reset_index(drop=True), aux, AUX_EDGE_TOL_K)
    m["aux_provenance"] = f"per-curve interpolation, edge tolerance {AUX_EDGE_TOL_K} K"
    del curves, pts
    print(f"matched table built in {time.time() - t0:.0f} s")
else:
    # ---- legacy fallback: an existing matched_points.csv (the B2/B3/B8 fixes cannot be applied without raw curves)
    p = Path("matched_points.csv")
    if not p.exists() and Path("seebeck_dataset.zip").exists():
        import zipfile; zipfile.ZipFile("seebeck_dataset.zip").extract("matched_points.csv", ".")
    if not p.exists() and IN_COLAB:
        from google.colab import files as _f; p = Path(next(iter(_f.upload())))
    m = pd.read_csv(p, low_memory=False)
    PROVENANCE["legacy_input"] = {"file": p.name, "sha256": sha256_file(p)}
    m["carrier_type"] = m["carrier_type"].replace({"sign-changing": "hall_sign_mixed_or_zero (legacy label)"})
    m["S_curve_uid"] = m["SID"].astype(str) + "_" + m["sample_id"].astype(str) + "_" + m["S_figure_id"].astype(str)
    m = m.sort_values(["S_curve_uid", "T_K"])
    g = m.assign(S_abs=m.S_VK.abs()).groupby("S_curve_uid")
    m["curve_points_original"] = g["T_K"].transform("size")
    i = g["S_abs"].idxmax(); pk = m.loc[i, ["S_curve_uid", "T_K"]].assign(Smax=m.loc[i, "S_VK"].abs().to_numpy())
    m = m.merge(pk.rename(columns={"T_K": "T_at_Smax"}), on="S_curve_uid", how="left")
    m["flag_postpeak_screen"] = (m.curve_points_original >= POSTPEAK_MIN_POINTS) & (m.T_K > m.T_at_Smax) & (m.S_VK.abs() < POSTPEAK_DROP * m.Smax)
    m["flag_seebeck_sign_mixed"] = m.groupby(["SID", "sample_id"])["S_VK"].transform(lambda v: (v > 0).any() and (v < 0).any())
    m["n_point_reuse"] = m.groupby(["SID", "sample_id", "n_figure_id", "T_n_K"])["T_K"].transform("size")
    m["aux_provenance"] = "legacy (sample-pooled, +/-5 K endpoint extension)"
    if "mu_m2Vs" not in m and "mu_source" in m:          # legacy file stores mobility in cm²/(V s) with its source
        m["mu_m2Vs"] = np.where(m["mu_source"].eq("reported"), m["mu_cm2Vs"] * 1e-4, np.nan)
    print("WARNING: legacy input mode; post-peak flag uses matched rows only and auxiliary interpolation is the legacy one.")

### 3.4 Composition, derived columns and QC flags
Nothing is removed here. `True` means a possible problem. Flags that use S (`flag_hall_seebeck_sign_mismatch`, `flag_seebeck_sign_mixed`, `flag_pf_sigma_S_inconsistent`) are for checking only.

In [ ]:
comp_cache = {c: parse_formula(c) for c in m["composition"].dropna().unique()}
desc = pd.DataFrame([{"composition": c, **composition_descriptors(f)} for c, f in comp_cache.items()])
m = m.drop(columns=[c for c in ("n_elements", "n_dopants", "dopant_total_frac", "el_n_valence_mean", "el_X_mean", "has_f_block")
                    if c in m.columns]).merge(desc, on="composition", how="left")
m["composition_status"] = m["composition_status"].fillna("unparsed_or_missing")
m["elem_frac_json"] = m["composition"].map(lambda c: json.dumps(comp_cache.get(c)) if comp_cache.get(c) else None)
m["S_uVK"] = m["S_VK"] * 1e6
m["S_abs_uVK"] = m["S_uVK"].abs()
m["n_cm3"] = m["n_m3"] * 1e-6
m["log10_n_cm3"] = np.log10(m["n_cm3"])
for c in ("sigma_Sm", "mu_m2Vs", "kappa_WmK", "leak_PF_WmK2", "leak_ZT"):
    if c not in m: m[c] = np.nan
mu_derived = m["sigma_Sm"] / (m["n_m3"] * E_CH)
m["mu_reported_over_derived"] = m["mu_m2Vs"] / mu_derived
m["log10_sigma_Sm"] = np.log10(m["sigma_Sm"])
m["S_SPB_abs_m1_uVK"] = spb_seebeck_abs_uVK(m["n_m3"], m["T_K"], 1.0)

m["flag_not_experiment"] = m["data_type"].ne("Experiment")
m["flag_not_bulk"] = ~m["form"].isin(["Bulk", "Polycrystal", "SingleCrystal", "OrientedBulk", "pellets", "disk"])
m["flag_S_range"] = m["S_uVK"].abs() > 1500
m["flag_n_range"] = ~m["n_cm3"].between(1e16, 1e23)
m["flag_T_range"] = ~m["T_K"].between(2, 1500)
sgn_expected = m["carrier_type"].map({"n": -1, "p": 1})
m["flag_hall_seebeck_sign_mismatch"] = sgn_expected.notna() & (np.sign(m["S_VK"]) != sgn_expected)
m["flag_hall_derived_n"] = m["n_source"].astype(str).str.startswith("hall")
m["flag_multiple_S_curves"] = m.groupby(["SID", "sample_id"])["S_curve_uid"].transform("nunique") > 1
with np.errstate(invalid="ignore"):
    m["flag_pf_sigma_S_inconsistent"] = (np.sqrt(m["leak_PF_WmK2"] / m["sigma_Sm"]) / m["S_VK"].abs() - 1).abs() > 0.3
m["flag_sigma_range"] = m["sigma_Sm"].notna() & ~m["sigma_Sm"].between(1e-2, 1e8)
m["flag_kappa_range"] = m["kappa_WmK"].notna() & ~m["kappa_WmK"].between(0.05, 500)
m["flag_mu_inconsistent"] = ~m["mu_reported_over_derived"].between(0.33, 3.0) & m["mu_reported_over_derived"].notna()
m["flag_sigma_endpoint_ext"] = m.get("sigma_Sm_endpoint_ext", pd.Series(False, index=m.index)).fillna(False).astype(bool)

m["study_id"] = m["SID"]
m["sample_key"] = m["SID"].astype(str) + "_" + m["sample_id"].astype(str)
m = m.sort_values(["SID", "sample_id", "S_curve_uid", "T_K"]).reset_index(drop=True)
m["point_id"] = np.arange(len(m))
m.to_csv(OUT / "data" / "matched_points.csv.gz", index=False)
if INPUT_MODE == "starrydata":
    save_table(pd.DataFrame(DLOG["unit_log"]).sort_values(["prop", "curves"], ascending=[True, False]), "unit_log", index=False)
    print("unit strings seen (accepted / rejected):"); display(pd.DataFrame(DLOG["unit_log"]))

ct_share = lambda d_: (d_["carrier_type"].eq("unknown").mean())
REPORT["matched"] = {"rows": len(m), "samples": int(m.sample_key.nunique()), "papers": int(m.SID.nunique()),
                     "carrier_type_rows": m["carrier_type"].value_counts().to_dict(),
                     "carrier_type_samples": m.drop_duplicates("sample_key")["carrier_type"].value_counts().to_dict(),
                     "unknown_share_rows": round(float(ct_share(m)), 4),
                     "unknown_share_samples": round(float(ct_share(m.drop_duplicates("sample_key"))), 4),
                     "S_n_pairing_dT_K": {"median": round(float(m.dT_K.median()), 3), "p95": round(float(m.dT_K.quantile(.95)), 3),
                                          "max": round(float(m.dT_K.max()), 3)},
                     "rows_with_reused_n_point": int((m.n_point_reuse > 1).sum()),
                     "n_source_rows": m["n_source"].value_counts().to_dict(),
                     "sigma_endpoint_extended_rows": int(m["flag_sigma_endpoint_ext"].sum()),
                     "seebeck_sign_mixed_samples": int(m.drop_duplicates("sample_key")["flag_seebeck_sign_mixed"].sum()),
                     "composition_status_rows": m["composition_status"].value_counts().to_dict(), **DLOG.get("raw_rows", {})}
print(json.dumps(REPORT["matched"], indent=1, default=str))

## 4. Eligibility domain (prespecified rules)
These rules are fixed **before** any split and **define the population every score refers to**. They are not tuned on results. Each removed row is logged with its first reason (`removed_rows.csv.gz`).

| Rule | Type | Note |
|---|---|---|
| composition parseable, known element symbols | data integrity | missing/unparsed JSON and unknown symbols are counted **separately** |
| experimental, bulk, broad range flags | data integrity | `DROP_FLAGS` |
| exclude `hall_sign_mixed` samples | physical scope | Hall-sign criterion. This does **not** remove every sample whose Seebeck sign changes (see `flag_seebeck_sign_mixed`) |
| `LIMITS` on T, n, \|S\|, composition means | physical scope | log₁₀ n ≤ 22.5, i.e. n ≤ 3.16 × 10²² cm⁻³. \|S\| from 5 to 1000 µV/K: **every result is conditional on this target range** |
| duplicates reused across papers | data integrity | same composition, T, n and S in a different paper |
| family exclusion, T ≥ `T_MIN`, conductivity available | physical scope / design | applied in Section 5 after classification |

The robust-z feature-outlier rule is **data-adaptive**, so it is not part of the domain. By default it is fitted on each fold's training papers and only removes training rows (Section 7). Rarity in electronegativity or valence is not evidence of bad data. Setting `FEATURE_OUTLIER_MODE="global"` reproduces the legacy behaviour (fitted on all rows, removes rows before splitting) as a documented alternative.

In [ ]:
d = m.copy()
d["remove_reason"] = ""
funnel = [("matched S–n points", len(d), d.sample_key.nunique(), d.SID.nunique())]
def mark(mask, reason):
    new = mask & d["remove_reason"].eq("")
    d.loc[new, "remove_reason"] = reason
    alive = d["remove_reason"].eq("")
    funnel.append((f"− {reason}", int(new.sum()), d.loc[alive, "sample_key"].nunique(), d.loc[alive, "SID"].nunique()))
    return int(new.sum())

mark(d["composition_status"].eq("unparsed_or_missing"), "composition missing or unparsed")
mark(d["composition_status"].eq("unknown_element_symbol"), "unknown element symbol (e.g. 'Ze', 'Zb')")
mark(d[DROP_FLAGS].any(axis=1), "QC flags " + "/".join(f.replace("flag_", "") for f in DROP_FLAGS))
# conductivity: blank values that failed QC (the row stays)
bad_sigma = d["flag_sigma_range"] | d["flag_mu_inconsistent"]
d.loc[bad_sigma, "log10_sigma_Sm"] = np.nan
if EXCLUDE_HALL_SIGN_MIXED:
    mark(d["carrier_type"].str.startswith("hall_sign_mixed"), "Hall coefficients of both signs in sample")
if EXCLUDE_HALL_ZERO_ONLY:
    mark(d["carrier_type"].eq("hall_zero_only"), "Hall coefficient only zero")
for c, (lo, hi) in LIMITS.items():
    mark(~d[c].between(lo, hi), f"limit {c} [{lo}, {hi}]")
if DEDUP_ACROSS_PAPERS:
    key = pd.DataFrame({"comp": d["composition"], "Tr": d["T_K"].round(0), "nr": d["log10_n_cm3"].round(2), "Sr": d["S_uVK"].round(0)})
    alive = d["remove_reason"].eq("")
    first_sid = d[alive].groupby([key.comp[alive], key.Tr[alive], key.nr[alive], key.Sr[alive]])["SID"].transform("first")
    dup = pd.Series(False, index=d.index); dup[alive] = key[alive].duplicated(keep="first") & (d.loc[alive, "SID"] != first_sid)
    mark(dup, "duplicate reused across papers")
if FEATURE_OUTLIER_MODE == "global":
    alive = d["remove_reason"].eq("")
    for c in OUTLIER_FEATURES:
        x = d.loc[alive, c]; med = x.median(); mad = (x - med).abs().median()
        if mad > 0: mark((0.6745 * (d[c] - med) / mad).abs() > ROBUST_Z, f"global robust-z outlier {c} (legacy)")

removed = d[d["remove_reason"].ne("")]
clean = d[d["remove_reason"].eq("")].drop(columns="remove_reason").reset_index(drop=True)
clean["log10_mu"] = np.log10(10 ** clean["log10_sigma_Sm"] / (clean["n_m3"] * E_CH) * 1e4)   # log10(μ / [1 cm²/(V s)]), μ = σ/(n_H e)
removed[["point_id", "SID", "sample_key", "composition", "T_K", "remove_reason"]].to_csv(OUT / "data" / "removed_rows.csv.gz", index=False)
print(f"clean: {len(clean)} points | {clean.sample_key.nunique()} samples | {clean.SID.nunique()} papers")
REPORT["clean"] = {"rows": len(clean), "samples": int(clean.sample_key.nunique()), "papers": int(clean.SID.nunique()),
                   "unknown_carrier_share_rows": round(float(clean.carrier_type.eq("unknown").mean()), 4),
                   "sigma_available_share": round(float(clean.log10_sigma_Sm.notna().mean()), 4),
                   "postpeak_flag_share": round(float(clean.flag_postpeak_screen.mean()), 4),
                   "log10_n_upper_bound": LIMITS["log10_n_cm3"][1], "n_upper_bound_cm3": float(10 ** LIMITS["log10_n_cm3"][1])}

## 5. Formula-compatible families, family exclusion and the final modelling domain
Labels come from the formula only. A family label means the formula is **compatible with** a structure type and its site rules. It does not establish the actual crystal structure or site occupancy (audit B5). The rules are unchanged from `mstar_calibration_5`, except:
* **Every** matching rule is now recorded (`family_candidates`, `n_family_candidates`, `family_ambiguous`), not only the first. Ambiguous compositions go to the top of the review list. An example is Ca0.995Na0.005Mg2Bi1.98, which matches both the Mg₃Pn₂ rule (Ca, Na as allowed substituents) and the CaAl₂Si₂-type AM₂Pn₂ rule.
* The marcasite family is named for what the rule actually matches: **Marcasite MPn2 (FeSb2-type)** (M = Fe, Ru, Os; Pn = Sb, As).
* **Exclusion (A5).** `FAMILY_EXCLUSION_MODE="host"` removes only compositions whose host label is FeSb2 (including substituted FeSb2), which is the stated intention. RuSb2 and FeAs2 stay in, and fall back to the global m* if their family has too few training papers. `"family"` restores the old whole-family rule. That rule needs its own physical justification, because a mass pinned at the search bound for the pooled family does not show that every member fails for the same reason.

Levels: **L1** verified table → **L2** constrained site rules (and composites (host)ₓ(additive)ᵧ with ≥ 90 % host) → **L3** broad chemical class (no own m*) → **L4** unclassified.

In [ ]:
# ---- element groups -------------------------------------------------------
CHALC = {"S", "Se", "Te"}
HALO = {"F", "Cl", "Br", "I"}
PNIC = {"N", "P", "As", "Sb", "Bi"}
TETREL = {"Si", "Ge", "Sn"}
ALKALI = {"Li", "Na", "K", "Rb", "Cs"}
AE = {"Be", "Mg", "Ca", "Sr", "Ba"}
RE = {"Sc", "Y", "La", "Ce", "Pr", "Nd", "Pm", "Sm", "Eu", "Gd", "Tb", "Dy", "Ho", "Er", "Tm", "Yb", "Lu"}
ACT = {"Th", "U", "Np", "Pu"}
TM = {"Ti","V","Cr","Mn","Fe","Co","Ni","Cu","Zn","Zr","Nb","Mo","Tc","Ru","Rh","Pd","Ag","Cd",
      "Hf","Ta","W","Re","Os","Ir","Pt","Au","Hg"}
POST = {"Al", "Ga", "In", "Tl", "Sn", "Pb", "Bi"}
METALS = ALKALI | AE | RE | ACT | TM | POST
ORGANIC = {"C", "H", "N"}

# ---- family definitions ---------------------------------------------------
# Each site: (name, host elements, allowed substituents, max substituent share OF THAT SITE)
# ratio: prototype site ratio (None for an optional filler/interstitial site: see 'filler')
# tol: max |site share - ideal share| (absolute, share of all atoms)
# filler: (site name, max filler atoms per 1 unit of reference site, reference site)
# host_fmt: how to write the host label from the majority element on each site
def _site(name, host, subs=(), max_sub=0.10):
    return (name, set(host), set(subs), max_sub)

RS_CAT_SUBS = {"Sn","Ge","Na","K","Ag","Cu","Sb","Bi","Tl","In","Ga","Mn","Cd","Mg","Ca","Sr","Ba","Eu","Yb","Hg","Zn","La","Ce","Cr","Li","Ni","Co","Fe","V","Ti","Sc","Y","Gd","Er","Dy","Nd","Pr","Sm"}
FAMILIES = [
 # --- chalcogenides ---
 dict(family="Tetradymite V2VI3 (Bi2Te3-type)", sites=[_site("V", {"Bi","Sb"}, {"In","Tl","Ga","Sn","Pb","Ge","Cu","Ag","Fe","Mn","Cr","Ni","Zn","Cd","Er","Ce","Sm","La","Lu","Yb","Gd","Dy","Ho","Na","K","Ti","Nb"}, 0.10),
                                                    _site("VI", {"Te","Se"}, {"S","I","Br","Cl"}, 0.10)],
      ratio=[2, 3], tol=0.05),
 dict(family="Rock-salt PbQ", sites=[_site("M", {"Pb"}, RS_CAT_SUBS, 0.45), _site("Q", {"Te","Se","S"}, {"I","Br","Cl"}, 0.05)], ratio=[1, 1], tol=0.045),
 dict(family="Rock-salt SnTe", sites=[_site("M", {"Sn"}, RS_CAT_SUBS - {"Sn"} | {"Pb"}, 0.45), _site("Q", {"Te"}, {"Se","S","I"}, 0.15)], ratio=[1, 1], tol=0.04),
 dict(family="Rhombohedral GeTe", sites=[_site("M", {"Ge"}, RS_CAT_SUBS - {"Ge"} | {"Pb"}, 0.45), _site("Q", {"Te"}, {"Se","S"}, 0.20)], ratio=[1, 1], tol=0.04),
 dict(family="Layered SnQ (SnSe/SnS, Pnma)", sites=[_site("M", {"Sn"}, {"Ge","Pb","Na","K","Ag","Cu","Sb","Bi","Zn","Cd","Ca","Sr","In","Tl","Li"}, 0.20), _site("Q", {"Se","S"}, {"Te","I","Br","Cl"}, 0.15)], ratio=[1, 1], tol=0.04),
 dict(family="I-V-VI2 rock-salt (AgSbTe2-type)", sites=[_site("I", {"Ag","Cu","Na","K"}, {"Li"}, 0.2), _site("V", {"Sb","Bi"}, {"In","Ga","Sn","Pb","Ge","Mn","Zn","Cd","Tl"}, 0.2), _site("VI", {"Te","Se"}, {"S"}, 0.2)],
      ratio=[1, 1, 2], tol=0.04),
 dict(family="Antifluorite/superionic M2Q (Cu2Se-type)", sites=[_site("M", {"Cu","Ag"}, {"Li","Na","K","Zn","Fe","Ni","Mn","Al","In","Sn","Pb","Bi","Sb","Cd","Hg","Co","Cr","Ga"}, 0.10), _site("Q", {"Se","S","Te"}, {"I","Br","Cl"}, 0.10)],
      ratio=[2, 1], tol=0.035),
 dict(family="Chalcopyrite I-III-VI2", sites=[_site("I", {"Cu","Ag"}, {"Na","Li","Zn","Cd","Mn"}, 0.2), _site("III", {"In","Ga","Al"}, {"Sb","Sn","Ge","Zn","Cd","Mn","Fe","Ni"}, 0.2), _site("VI", {"Te","Se","S"}, set(), 0)],
      ratio=[1, 1, 2], tol=0.04),
 dict(family="Diamond-like I2-IV-VI3 (Cu2SnSe3-type)", sites=[_site("I", {"Cu","Ag"}, {"Zn","Cd","Mn","Fe","In","Ga","Ni","Co"}, 0.2), _site("IV", {"Sn","Ge"}, {"In","Ga","Sb","Si","Zn","Mn"}, 0.2), _site("VI", {"Se","S","Te"}, set(), 0)],
      ratio=[2, 1, 3], tol=0.035),
 dict(family="Diamond-like I3-V-VI4 (Cu3SbSe4-type)", sites=[_site("I", {"Cu","Ag"}, {"Zn","Cd","Mn","Fe","Ni"}, 0.15), _site("V", {"Sb","As"}, {"Sn","Ge","Bi","Al","Ga","In"}, 0.25), _site("VI", {"Se","S","Te"}, set(), 0)],
      ratio=[3, 1, 4], tol=0.03),
 dict(family="Diamond-like I2-II-IV-VI4 (kesterite/stannite)", sites=[_site("I", {"Cu","Ag"}, {"Na","Li"}, 0.15), _site("II", {"Zn","Cd","Fe","Mn","Co","Ni","Mg","Hg"}, {"Cu","In"}, 0.25), _site("IV", {"Sn","Ge","Si"}, {"In","Ga","Sb"}, 0.25), _site("VI", {"Se","S","Te"}, set(), 0)],
      ratio=[2, 1, 1, 4], tol=0.03),
 dict(family="Argyrodite A8MQ6", sites=[_site("A", {"Ag","Cu"}, {"Li","Na"}, 0.15), _site("M", {"Sn","Ge","Si"}, {"Ga","Sb"}, 0.25), _site("Q", {"Se","S","Te"}, {"I","Br","Cl"}, 0.2)],
      ratio=[8, 1, 6], tol=0.03),
 dict(family="In4Q3 (In4Se3-type)", sites=[_site("M", {"In"}, {"Sn","Pb","Ga","Cu","Ag","Na","Tl"}, 0.1), _site("Q", {"Se","Te"}, {"S","Cl","I"}, 0.1)], ratio=[4, 3], tol=0.035),
 dict(family="BiCuSeO oxychalcogenide", sites=[_site("Bi", {"Bi"}, {"Pb","Ba","Ca","Sr","Mg","La","Na","K","Ag","Zn","Cd","Sn","Sb","Ce","Nd","Sm","Yb","Ti","Zr"}, 0.2), _site("Cu", {"Cu"}, {"Ag","Ni","Fe","Co","Zn"}, 0.15),
                                         _site("Q", {"Se"}, {"Te","S"}, 0.5), _site("O", {"O"}, {"F"}, 0.1)], ratio=[1, 1, 1, 1], tol=0.04),
 # --- pnictides / Zintl ---
 dict(family="Skutterudite MPn3", sites=[_site("M", {"Co","Rh","Ir","Fe","Ru","Os"}, {"Ni","Pd","Pt","Cr","Mn","Cu"}, 0.3), _site("Pn", {"Sb","As","P"}, {"Te","Sn","Ge","Se","Bi","S"}, 0.20),
                                        _site("R", set(), RE | AE | ALKALI | ACT | {"In","Ga","Tl","Eu","Yb"}, 1.0)],
      ratio=[1, 3, None], tol=0.04, filler=("R", 0.40, "M")),
 dict(family="Anti-La2O3 Mg3Pn2 Zintl", sites=[_site("A", {"Mg"}, {"Ca","Yb","Mn","Zn","Cd","Na","Li","Ag","Sc","Y","La","Nb","Hf","Ta","Fe","Co","Ce","Pr","Nd","Sm","Gd","Er","Ti","Cu"}, 0.35), _site("Pn", {"Sb","Bi"}, {"Te","Se","S","As","P","Sn","Ge"}, 0.05)],
      ratio=[3, 2], tol=0.04),
 dict(family="CaAl2Si2-type AM2Pn2 Zintl", sites=[_site("A", {"Ca","Sr","Ba","Eu","Yb"}, {"Na","K","Mg","La","Ce","Nd","Sm"}, 0.25), _site("M", {"Zn","Cd","Mn","Mg","Al"}, {"Cu","Ag","Fe","Co","Ni"}, 0.25), _site("Pn", {"Sb","As","Bi","P"}, {"Te","Sn","Ge"}, 0.05)],
      ratio=[1, 2, 2], tol=0.035),
 dict(family="Yb14MnSb11-type A14MPn11 Zintl", sites=[_site("A", {"Yb","Ca","Sr","Eu"}, {"La","Ce","Pr","Nd","Na"}, 0.25), _site("M", {"Mn","Mg","Zn","Al"}, {"Cd"}, 0.3), _site("Pn", {"Sb","Bi","As"}, {"Te","Ge"}, 0.05)],
      ratio=[14, 1, 11], tol=0.02),
 dict(family="Zintl chain/ribbon antimonides (A9M4Pn9, A5M2Pn6, A3MPn3)", sites=[_site("A", {"Yb","Ca","Sr","Eu"}, {"Na","La","Ce"}, 0.25), _site("M", {"Mn","Zn","Cd","Al","Ga","In"}, {"Mg"}, 0.3), _site("Pn", {"Sb","Bi","As"}, set(), 0)],
      ratio=[9, 4.2, 9], tol=0.035, alt_ratios=[[5, 2, 6], [3, 1, 3]]),
 dict(family="ZrBeSi/LiGaGe-type ABPn (BaAgSb, EuCuSb)", sites=[_site("A", {"Ca","Sr","Ba","Eu","Yb"}, {"Ce","La","Na","K"}, 0.25), _site("B", {"Ag","Cu","Zn","Au"}, {"Ni","Cd","Mn","Li"}, 0.2), _site("Pn", {"Sb","As","Bi","P"}, {"Te","Sn","Ge"}, 0.1)],
      ratio=[1, 1, 1], tol=0.035, alt_ratios=[[1, 0.6, 1], [1, 0.8, 1]]),
 dict(family="alpha-MgAgSb", sites=[_site("Mg", {"Mg"}, {"Na","Li","Ca","Yb"}, 0.1), _site("Ag", {"Ag"}, {"Cu","Zn","Ni"}, 0.1), _site("Sb", {"Sb"}, {"Bi","Te","Sn","In"}, 0.1)], ratio=[1, 1, 1], tol=0.03),
 dict(family="Zn-Sb (ZnSb / Zn4Sb3)", sites=[_site("M", {"Zn","Cd"}, {"Ag","Cu","Ga","In","Mg","Pb","Sn","Ni","Te"}, 0.1), _site("Sb", {"Sb"}, {"Bi","Te","Sn","Ge"}, 0.1)],
      ratio=[1, 1], tol=0.03, alt_ratios=[[4, 3], [13, 10]]),
 dict(family="Marcasite MPn2 (FeSb2-type)", sites=[_site("M", {"Fe","Ru","Os"}, {"Co","Cr","Ni","Mn"}, 0.2), _site("Pn", {"Sb","As"}, {"Te","Sn","Bi","Se"}, 0.15)], ratio=[1, 2], tol=0.03),
 # --- tetrelides / intermetallics ---
 dict(family="Half-Heusler XYZ", sites=[_site("X", {"Ti","Zr","Hf","V","Nb","Ta","Sc","Y","Er","Dy","Gd","Ho","Lu","Tm","Yb","Mg","Li"}, {"Ce","La","Sm","Nd","Mo","W"}, 0.1),
                                       _site("Y", {"Ni","Co","Fe","Pd","Pt","Rh","Ir","Ru","Cu","Mn"}, {"Cr","Au","Ag","Os"}, 0.1),
                                       _site("Z", {"Sn","Sb","Bi","Ge","Si","Pb","In","Ga","Al"}, {"Te"}, 0.05)],
      ratio=[1, 1, 1], tol=0.045),
 dict(family="Antifluorite Mg2X (Mg2Si-Sn-Ge)", sites=[_site("Mg", {"Mg"}, {"Li","Na","K","Ag","Ca","Mn","Zn","Cu"}, 0.1), _site("X", {"Si","Ge","Sn"}, {"Sb","Bi","Al","Ga","In","Pb","P","As"}, 0.1)],
      ratio=[2, 1], tol=0.035),
 dict(family="Si-Ge alloy (diamond)", sites=[_site("SiGe", {"Si","Ge"}, {"P","B","Ga","Al","As","Sb","Na","Au","Ni","Fe","C","Mo","W","Y"}, 0.08)], ratio=[1], tol=1.0),
 dict(family="Bi-Sb alloy (semimetal)", sites=[_site("BiSb", {"Bi","Sb"}, {"Sn","Te","Pb","Ga","In","Ge","Se"}, 0.05)], ratio=[1], tol=1.0),
 dict(family="B20 CoSi-type", sites=[_site("M", {"Co","Fe","Mn","Cr"}, {"Ni","Rh","Ru","Pd","Pt","Ir"}, 0.25), _site("Si", {"Si","Ge"}, {"Al","Ga","B","P","Sn"}, 0.1)], ratio=[1, 1], tol=0.04),
 dict(family="Higher manganese silicide MnSi~1.75", sites=[_site("Mn", {"Mn"}, {"Cr","Fe","Re","Mo","V","Co","Ru"}, 0.2), _site("Si", {"Si"}, {"Ge","Al","Ga","Sn","B"}, 0.15)], ratio=[1, 1.74], tol=0.03),
 # --- oxides ---
 dict(family="Perovskite SrTiO3", sites=[_site("A", {"Sr"}, {"La","Nd","Sm","Gd","Pr","Ba","Ca","Y","Dy","Eu","Ce","Er","Yb","Bi","Na","K"}, 0.35), _site("B", {"Ti"}, {"Nb","Ta","Co","Fe","V","Mn","Cr","Zr","Mo","W"}, 0.35), _site("O", {"O"}, {"F","N"}, 0.05)],
      ratio=[1, 1, 3], tol=0.04),
 dict(family="Perovskite ABO3 (other)", sites=[_site("A", {"La","Pr","Nd","Sm","Ca","Sr","Ba","Bi","Y","Gd","Eu","Ce","Na","K"}, set(), 0), _site("B", {"Mn","Fe","Co","Ni","Cr","Ti","V","Ru","Rh","Cu","Nb","Ta","Zr","Sn","In","Ga","Al","Mo","W"}, set(), 0), _site("O", {"O"}, {"F"}, 0.05)],
      ratio=[1, 1, 3], tol=0.04),
 dict(family="Wurtzite ZnO", sites=[_site("Zn", {"Zn"}, {"Al","Ga","In","Ni","Mg","Co","Ti","Mn","Fe","Cu","Sb","Bi","Cd","Sn","Y","Zr","Li","Na"}, 0.10), _site("O", {"O"}, {"F","N","S"}, 0.05)], ratio=[1, 1], tol=0.04),
 dict(family="Layered NaxCoO2", sites=[_site("A", {"Na"}, {"K","Ca","Sr","Ag","Li","Bi","La","Yb"}, 0.4), _site("Co", {"Co"}, {"Fe","Ni","Cu","Mn","Rh","Zn","Ru","Pd","Pt","Sb","Ti"}, 0.2), _site("O", {"O"}, set(), 0)],
      ratio=[0.7, 1, 2], tol=0.06),
 dict(family="Misfit Ca3Co4O9", sites=[_site("Ca", {"Ca"}, {"Sr","Bi","Na","La","Lu","Y","Gd","Ag","Ba","Yb","Dy","Nd","Pb"}, 0.4), _site("Co", {"Co"}, {"Fe","Ni","Cu","Mn","Rh","Ti","Cr","Ga","Zn","Mo","Sb","Pd","W","Nb"}, 0.25), _site("O", {"O"}, set(), 0)],
      ratio=[3, 4, 9], tol=0.04),
]

# known SPB caveats (family-level, informational only)
SPB_CAVEAT = {
 "Bi-Sb alloy (semimetal)": "semimetal / narrow gap: bipolar, SPB not expected to hold",
 "Marcasite MPn2 (FeSb2-type)": "correlated semiconductor, phonon drag at low T",
 "Layered NaxCoO2": "correlated oxide (Heikes-type S), SPB not expected to hold",
 "Misfit Ca3Co4O9": "correlated oxide (Heikes-type S), SPB not expected to hold",
 "B20 CoSi-type": "semimetal",
}

# ---- matching engine --------------------------------------------------------
def _site_of(el, sites):
    for i, (_, host, subs, _) in enumerate(sites):
        if el in host: return i, True
    for i, (_, host, subs, _) in enumerate(sites):
        if el in subs: return i, False
    return None, None

def _fmt(x):
    return "" if abs(x - 1) < 1e-9 else (f"{x:g}")

def match_family(f, spec):
    """Returns dict(host, family, exact_host, dev, subs) or None."""
    sites = spec["sites"]
    tot = [0.0] * len(sites); host_amt = [0.0] * len(sites); per_el = [dict() for _ in sites]
    for el, v in f.items():
        i, is_host = _site_of(el, sites)
        if i is None:
            return None                                     # element not allowed anywhere
        tot[i] += v; per_el[i][el] = v
        if is_host: host_amt[i] += v
    # every non-filler site must be occupied and dominated by host elements
    for i, (nm, host, subs, max_sub) in enumerate(sites):
        if spec["ratio"][i] is None: continue
        if tot[i] <= 0 or (host and host_amt[i] <= 0): return None
        if host and (tot[i] - host_amt[i]) / tot[i] > max_sub + 1e-9: return None
    # stoichiometry (main sites only; filler handled separately)
    main = [i for i, r in enumerate(spec["ratio"]) if r is not None]
    T = sum(tot[i] for i in main)
    best = None
    for ratio in [spec["ratio"]] + [r + [None] * (len(sites) - len(r)) for r in spec.get("alt_ratios", [])]:
        R = sum(ratio[i] for i in main)
        dev = max(abs(tot[i] / T - ratio[i] / R) for i in main)
        if best is None or dev < best[0]: best = (dev, ratio)
    dev, ratio = best
    if dev > spec["tol"]: return None
    if "filler" in spec:
        fs, fmax, ref = spec["filler"]
        fi = [s[0] for s in sites].index(fs); ri = [s[0] for s in sites].index(ref)
        if tot[fi] / tot[ri] > fmax: return None
    elif any(tot[i] > 0 for i, r in enumerate(spec["ratio"]) if r is None):
        return None
    # host label: majority host element on each main site, with the prototype ratio
    parts = []
    for i in main:
        maj = max(per_el[i], key=per_el[i].get)
        if len(sites) == 1:
            parts = ["-".join(sorted(e for e in per_el[i] if e in sites[i][1]))]; break
        parts.append(maj + _fmt(ratio[i]))
    host = "".join(parts)
    if "filler" in spec and tot[[s[0] for s in sites].index(spec["filler"][0])] > 0:
        host = "filled " + host
    subs = sorted(e for i, s in enumerate(sites) for e in per_el[i] if e not in s[1])
    return dict(host=host, family=spec["family"], exact_host=(not subs and dev <= 0.02), dev=round(dev, 4), subs=",".join(subs))

# ---- broad chemical fallback (always computed) ----------------------------------
def broad_class(f):
    g = lambda s: sum(v for e, v in f.items() if e in s)
    o, ch, hal, pn, tet = f.get("O", 0), g(CHALC), g(HALO), g(PNIC), g(TETREL)
    metal = g(METALS)
    if g(ORGANIC) >= 0.3 and f.get("C", 0) >= 0.1:
        return "Carbon / organic"
    if o >= 0.10:
        if ch >= 0.05: return "Mixed-anion: oxychalcogenide"
        if hal >= 0.05: return "Mixed-anion: oxyhalide"
        if pn - f.get("Bi", 0) >= 0.05: return "Mixed-anion: oxypnictide"
        return "Oxide"
    if ch >= 0.90 and metal < 0.05:
        return "Elemental chalcogen (Te/Se)"
    if ch >= 0.20:
        if hal >= 0.05: return "Mixed-anion: chalcohalide"
        shares = {e: f.get(e, 0) / ch for e in CHALC}
        major = [e for e, s in shares.items() if s >= 0.2]
        if len(major) > 1: return "Mixed chalcogenide (" + "/".join(sorted(major)) + ")"
        return {"Te": "Telluride", "Se": "Selenide", "S": "Sulfide"}[max(shares, key=shares.get)]
    if ch >= 0.05 and pn >= 0.20:
        return "Mixed-anion: pnictide-chalcogenide"
    nonmetal_frac = 1 - metal
    if set(f) <= {"Bi", "Sb", "Te", "Se", "Si", "Ge", "Sn", "Pb", "As", "B", "P", "C"} and metal - g(POST) < 0.05:
        return "Elemental / metalloid alloy"
    if pn - f.get("Bi", 0) >= 0.20 and g(ALKALI | AE | RE | TM | ACT) >= 0.2:
        return "Pnictide (" + max(("Sb", "As", "P", "N"), key=lambda e: f.get(e, 0)) + ")"
    if tet >= 0.20 and g(TM | RE | AE | ALKALI | ACT) >= 0.2:
        return "Tetrelide (silicide/germanide/stannide)"
    if metal >= 0.75:
        return "Intermetallic (rare-earth / actinide)" if g(RE | ACT) >= 0.05 else "Intermetallic"
    return "Unclassified"

# ---- composites written as (host)x(additive)y ------------------------------------
# Only formulas that explicitly state components are treated as composites. The largest component must
# carry >= COMPOSITE_MIN of the atoms and must itself match a family; the additive is recorded, not hidden.
COMPOSITE_MIN = 0.90
_TOP = re.compile(r"\(([^()]*(?:\([^()]*\)[^()]*)*)\)(\d*\.?\d*)|([A-Z][a-z]?\d*\.?\d*)")
def split_components(formula, parse):
    s = re.sub(r"[\s\-·•]", "", formula or "")
    if "(" not in s: return None
    comps, pos, bare = [], 0, ""
    for m_ in _TOP.finditer(s):
        if m_.start() != pos: return None
        pos = m_.end()
        if m_.group(1) is not None:
            if bare: comps.append((bare, 1.0)); bare = ""
            comps.append((m_.group(1), float(m_.group(2)) if m_.group(2) else 1.0))
        else:
            bare += m_.group(3)
    if bare: comps.append((bare, 1.0))
    if pos != len(s) or len(comps) < 2: return None
    out = []
    for sub, k in comps:
        f = parse(sub)
        if not f: return None
        atoms = k * _atoms(sub)
        out.append((sub, f, atoms))
    return out
def _atoms(sub):
    tot = 0.0
    for el, num in re.findall(r"([A-Z][a-z]?)(\d*\.?\d*)", sub):
        tot += float(num) if num else 1.0
    return tot

COMPOSITE_MIN = COMPOSITE_MIN_SHARE

def classify(comp, frac, verified):
    b = broad_class(frac) if frac else "Unparsed"
    if verified and comp in verified:
        h, fam = verified[comp]
        return dict(host=h, family=fam, broad_class=b, class_level="L1 verified", exact_host=False, subs="",
                    spb_caveat=SPB_CAVEAT.get(fam, ""), family_candidates=fam, n_family_candidates=1)
    cands = []
    if frac:
        cands = [(r, "L2 constrained_formula") for spec in FAMILIES if (r := match_family(frac, spec))]
        if not cands:
            parts = split_components(comp, parse_formula)
            if parts:
                tot = sum(a for _, _, a in parts); main_sub, main_f, main_a = max(parts, key=lambda t: t[2])
                if main_a / tot >= COMPOSITE_MIN:
                    extra = "; ".join(p_ for p_, _, _ in parts if p_ != main_sub)
                    for spec in FAMILIES:
                        r = match_family(main_f, spec)
                        if r:
                            r["subs"] = r["subs"] + ("; " if r["subs"] else "") + f"second phase: {extra}"; r["exact_host"] = False
                            cands.append((r, "L2 constrained_formula (composite)"))
    if cands:
        r, lvl = cands[0]
        return dict(host=r["host"], family=r["family"], broad_class=b, class_level=lvl, exact_host=r["exact_host"],
                    subs=r["subs"], spb_caveat=SPB_CAVEAT.get(r["family"], ""),
                    family_candidates=" | ".join(c[0]["family"] for c in cands), n_family_candidates=len(cands))
    lvl = "L4 unclassified" if b in ("Unclassified", "Unparsed") else "L3 broad_fallback"
    return dict(host=None, family=None, broad_class=b, class_level=lvl, exact_host=False, subs="", spb_caveat="",
                family_candidates="", n_family_candidates=0)

verified = {}
if Path(VERIFIED_TABLE).exists():
    v = pd.read_csv(VERIFIED_TABLE).dropna(subset=["family"])
    verified = {r.composition: (r.host, r.family) for r in v.itertuples()}
    print(f"loaded {len(verified)} verified compositions from {VERIFIED_TABLE}")
cls = {c: classify(c, comp_cache.get(c), verified) for c in clean["composition"].unique()}
for k in ["host", "family", "broad_class", "class_level", "exact_host", "subs", "spb_caveat", "family_candidates", "n_family_candidates"]:
    clean[k] = clean["composition"].map(lambda c: cls[c][k])
clean["family_ambiguous"] = clean["n_family_candidates"] > 1
clean["group"] = clean["family"].where(clean["family"].notna(), "[fallback] " + clean["broad_class"])
REPORT["families"] = {"family_rules_defined": len(FAMILIES),
                      "families_represented_clean": int(clean["family"].nunique()),
                      "fallback_groups_clean": int(clean.loc[clean.family.isna(), "group"].nunique()),
                      "ambiguous_compositions": int(clean.drop_duplicates("composition")["family_ambiguous"].sum())}
print("classification level (points / samples / papers):")
display(clean.groupby("class_level").agg(points=("point_id", "size"), samples=("sample_key", "nunique"), papers=("SID", "nunique")))
amb = clean[clean.family_ambiguous].drop_duplicates("composition")[["composition", "SID", "family", "family_candidates"]]
print(f"{len(amb)} compositions match more than one family rule (first match used, all recorded):"); display(amb.head(15))

In [ ]:
# ---- family exclusion (A5), then the remaining domain rules
d2 = clean.copy(); d2["remove_reason"] = ""
def mark2(mask, reason):
    new = mask & d2["remove_reason"].eq(""); d2.loc[new, "remove_reason"] = reason
    alive = d2["remove_reason"].eq("")
    funnel.append((f"− {reason}", int(new.sum()), d2.loc[alive, "sample_key"].nunique(), d2.loc[alive, "SID"].nunique()))
funnel.append(("= clean (Section 4)", len(clean), clean.sample_key.nunique(), clean.SID.nunique()))
if FAMILY_EXCLUSION_MODE == "host":
    excl = d2["host"].isin(EXCLUDE_HOSTS)
    mark2(excl, f"excluded host {EXCLUDE_HOSTS}")
elif FAMILY_EXCLUSION_MODE == "family":
    excl = d2["family"].isin(EXCLUDE_FAMILIES)
    mark2(excl, f"excluded whole family {EXCLUDE_FAMILIES}")
else:
    excl = pd.Series(False, index=d2.index)
exc_tab = (clean[excl].groupby(["host", "composition"]).agg(SID=("SID", lambda s: sorted(set(s))), points=("point_id", "size"))
           .reset_index().sort_values("points", ascending=False))
print(f"excluded by the family rule ({FAMILY_EXCLUSION_MODE}):", int(excl.sum()), "points"); display(exc_tab.head(20))
kept_marc = clean[~excl & clean["family"].eq("Marcasite MPn2 (FeSb2-type)")].groupby("host").agg(points=("point_id", "size"), SIDs=("SID", lambda s: sorted(set(s))))
if len(kept_marc): print("marcasite-rule members that are KEPT:"); display(kept_marc)
mark2(d2["T_K"] < T_MIN, f"T < {T_MIN} K")
if REQUIRE_MOBILITY:
    mark2(d2["log10_mu"].isna(), "no QC-passed conductivity")
data = d2[d2.remove_reason.eq("")].drop(columns="remove_reason").reset_index(drop=True)
funnel.append(("= final modelling domain", len(data), data.sample_key.nunique(), data.SID.nunique()))
funnel_df = pd.DataFrame(funnel, columns=["step", "points (removed or total)", "samples remaining", "papers remaining"])
save_table(funnel_df, "domain_funnel", index=False); display(funnel_df)
DOMAIN = (f"|S| ∈ [{LIMITS['S_abs_uVK'][0]:g}, {LIMITS['S_abs_uVK'][1]:g}] µV/K, T ∈ [{max(T_MIN, LIMITS['T_K'][0]):g}, "
          f"{LIMITS['T_K'][1]:g}] K, n ∈ [1e{LIMITS['log10_n_cm3'][0]:g}, 1e{LIMITS['log10_n_cm3'][1]:g}] cm⁻³, bulk experimental, "
          f"conductivity required, Hall-sign-mixed samples excluded, excluded: {EXCLUDE_HOSTS if FAMILY_EXCLUSION_MODE == 'host' else EXCLUDE_FAMILIES if FAMILY_EXCLUSION_MODE == 'family' else 'none'}")
print("DOMAIN:", DOMAIN)
REPORT["final_domain"] = {"rows": len(data), "samples": int(data.sample_key.nunique()), "papers": int(data.SID.nunique()),
                          "unknown_carrier_share_rows": round(float(data.carrier_type.eq("unknown").mean()), 4),
                          "domain": DOMAIN, "excluded_by_family_rule_points": int(excl.sum()),
                          "excluded_compositions": exc_tab.assign(SID=exc_tab.SID.astype(str)).to_dict(orient="records")}

# review table: ambiguous and fallback labels carrying the most data first
review = (data.groupby("composition").agg(host=("host", "first"), family=("family", "first"), family_candidates=("family_candidates", "first"),
          ambiguous=("family_ambiguous", "first"), broad_class=("broad_class", "first"), class_level=("class_level", "first"),
          points=("point_id", "size"), samples=("sample_key", "nunique"), papers=("SID", "nunique")).reset_index())
unc = {"L3 broad_fallback": 3, "L4 unclassified": 3, "L2 constrained_formula (composite)": 2, "L2 constrained_formula": 1, "L1 verified": 0}
review["priority"] = review["points"] * (review["class_level"].map(unc).fillna(1) + 2 * review["ambiguous"])
review = review.sort_values("priority", ascending=False).assign(verified_host="", verified_family="")
save_table(review, "family_review_priority", index=False)

## 6. Papers reserved after exploratory analysis, and development folds
**What these papers are (A1).** Every one of the 41 papers reserved on 24 Sep 2026 was used in the *earlier* exploratory notebooks. Out-of-fold calibration predictions exist for all of their rows, and feature screening, family and pooling decisions and domain restrictions were developed on data that included them. Reserving them afterwards cannot undo that exposure, and the size of any resulting optimism is **unknown** (not "small"). They are therefore described as **papers reserved for subsequent evaluation after exploratory analysis**, not as an untouched test set.

What this notebook does about it:
1. It keeps the same papers (`RESERVED_MODE="fixed_list"`), so they are not re-chosen after seeing results. Re-salting and choosing other already-used papers would not create independence.
2. It **seals** them: every later section uses development papers only. That covers mass fits, K-means, transfer trees, the input decision and all descriptive fits. The sealed frame can only be opened in Section 14 with an explicit confirmation phrase.
3. A genuinely independent validation needs **new papers** (e.g. those added to Starrydata after this release), or an outer evaluation in which the whole selection process is repeated inside training data.

**Development folds (C2).** With `FOLD_METHOD="shuffled_greedy"`, papers are ordered only by SHA-256(`seed-SID`) and then balanced greedily, so seeds give genuinely different partitions. The table below shows how many papers keep the same fold label across all seeds. The SD over seeds describes **sensitivity to the split rule**. It is not a standard error over 25 independent experiments.

In [ ]:
if RESERVED_MODE == "fixed_list":
    reserved_sids = sorted(set(RESERVED_SIDS_PREVIOUS) & set(data.SID))
    missing = sorted(set(RESERVED_SIDS_PREVIOUS) - set(data.SID))
    if missing: print(f"{len(missing)} previously reserved papers are not in the current domain (kept out anyway): {missing}")
else:
    stratum = data.groupby("SID")["group"].agg(lambda s: s.value_counts().idxmax())
    reserved_sids = []
    for g_, sids in stratum.groupby(stratum):
        sids = sorted(sids.index, key=lambda s_: sha(f"{TEST_SALT}-{s_}")); npap = len(sids)
        k_ = 0 if npap < 3 else max(int(round(RESERVED_FRACTION * npap)), 1 if npap >= 5 else 0)
        reserved_sids += sids[:k_]
    reserved_sids = sorted(int(s_) for s_ in reserved_sids)

class Sealed:
    """Holds the reserved rows. Only counts are visible; opening requires the confirmation phrase."""
    def __init__(self, df): self._df = df
    def __repr__(self): return f"<sealed: {len(self._df)} rows, {self._df.SID.nunique()} papers>"
    def open(self, confirm):
        if confirm != CONFIRM_PHRASE: raise PermissionError("Reserved papers are sealed during development.")
        return self._df.copy()
RESERVED = Sealed(data[data.SID.isin(reserved_sids)].reset_index(drop=True))
dev = data[~data.SID.isin(reserved_sids)].reset_index(drop=True)
assert not set(dev.SID) & set(reserved_sids) and not set(dev.sample_key) & set(data[data.SID.isin(reserved_sids)].sample_key)
print("development:", len(dev), "rows |", dev.sample_key.nunique(), "samples |", dev.SID.nunique(), "papers;  reserved:", RESERVED)

# reserved share per group: approximate by construction (majority group per paper, integer rounding, small groups get none)
by_g = (data.assign(split=np.where(data.SID.isin(reserved_sids), "reserved", "dev")).groupby(["group", "split"])["SID"].nunique()
        .unstack(fill_value=0).reindex(columns=["dev", "reserved"], fill_value=0).assign(total=lambda x: x.sum(axis=1))
        .sort_values("total", ascending=False))
save_table(by_g, "reserved_papers_by_group")

for sd in SEEDS:
    dev[f"fold_s{sd}"] = make_folds(dev, N_FOLDS, sd)
    assert (dev.groupby("SID")[f"fold_s{sd}"].nunique() == 1).all()
PRIMARY = SEEDS[0]
fcols = [f"fold_s{sd}" for sd in SEEDS]
per_paper = dev.groupby("SID")[fcols].first(); rows_pp = dev.groupby("SID").size()
same = per_paper.nunique(axis=1).eq(1)
# co-assignment stability: share of paper pairs that share a fold under seed A and also under seed B (chance ≈ 1/N_FOLDS)
co = []
for i, a in enumerate(fcols):
    for b in fcols[i + 1:]:
        A = per_paper[a].to_numpy()[:, None] == per_paper[a].to_numpy()[None, :]
        B = per_paper[b].to_numpy()[:, None] == per_paper[b].to_numpy()[None, :]
        iu = np.triu_indices(len(A), 1); co.append((A[iu] & B[iu]).sum() / A[iu].sum())
fold_stab = {"fold_method": FOLD_METHOD, "papers_same_fold_all_seeds": int(same.sum()), "papers": len(same),
             "rows_in_those_papers": int(rows_pp[same].sum()), "rows": len(dev),
             "mean_pair_coassignment_across_seeds": round(float(np.mean(co)), 3), "chance_level": round(1 / N_FOLDS, 3)}
REPORT["folds"] = fold_stab; REPORT["reserved"] = {"papers": len(reserved_sids), "SIDs": reserved_sids,
    "rows": len(data) - len(dev), "mode": RESERVED_MODE, "status": "reserved after exploratory analysis; not untouched"}
print(json.dumps(fold_stab, indent=1))
display(dev.groupby(f"fold_s{PRIMARY}").agg(points=("point_id", "size"), samples=("sample_key", "nunique"), papers=("SID", "nunique")))

fig, ax = plt.subplots(figsize=(11, 0.33 * len(by_g) + 1.4))
y = np.arange(len(by_g))[::-1]
ax.barh(y, by_g["dev"], color=CAT[0], height=0.62, label="development")
ax.barh(y, by_g["reserved"], left=by_g["dev"], color=CAT[1], height=0.62, label="reserved after exploration")
ax.set_yticks(y, [g_.replace("[fallback] ", "▸ ")[:55] for g_ in by_g.index])
ax.set_xlabel("papers"); ax.xaxis.grid(True, color=GRID, lw=0.6); ax.set_axisbelow(True); ax.legend(frameon=False, loc="lower right")
ax.set_title(f"Papers per group: {len(reserved_sids)} reserved of {data.SID.nunique()}", loc="left")
plt.tight_layout(); save_fig(fig, "fig_reserved_split"); plt.show()

## 7. Fold engine: training filters → weights → m* calibration → dimensionless features
For every seed × fold, the same steps run on the **training papers only**, and all models reuse the resulting frames, so every comparison is exactly paired (same rows, folds, weights and masses):

1. **Training-only filters.** Post-peak screened points are removed (`TRAIN_EXCLUDE_POSTPEAK`). Robust-z feature outliers are removed with median/MAD fitted on this fold's training rows (`FEATURE_OUTLIER_MODE="train_fold"`). Validation rows are never filtered.
2. **Weights (A2)** are computed **after** these filters: $w = 1/(\text{retained training points of the sample})$, so every retained sample carries total weight 1. The legacy policy (weights set before filtering, so a sample's weight is its retained fraction) is available as `WEIGHTING="retained_fraction"` and is run as a sensitivity check.
3. **m* calibration** on the filtered, weighted training rows. The loss is the weighted mean of $\big|\,|S|-S_{\rm SPB}\big|$ (magnitudes; the signed S is never the fitted target).
4. **Dimensionless features** with the fold's masses: $s=|S|/(k_B/e)$, $s_{\rm SPB}$, $\ln(n/N_c)$, $\tau=T/300\,$K, $\log_{10}(\mu/[1\,\text{cm}^2/(\text{V s})])$ with $\mu=\sigma/(n_H e)$, mean valence. The target is $\Delta s = s - s_{\rm SPB}$.

"Hall-only" means n, T **and the composition-derived family label**, which picks the family mass. It does not mean Hall measurements alone.

In [ ]:
FAMS_ALL = sorted(data["group"].unique())
def prepare(df):
    df = df.copy()
    df["s"] = df["S_abs_uVK"] / KBE_UV
    df["tau"] = df["T_K"] / T_REF
    df["val_mean"] = df["el_n_valence_mean"]
    df["w_pre"] = 1.0 / df.groupby("sample_key")["sample_key"].transform("size")      # legacy: weight set before filtering
    for g_ in FAMS_ALL:                                                                # target-free label one-hot (raw-input baseline)
        df[f"fam__{g_}"] = (df["group"] == g_).astype(float)
    return df
dev = prepare(dev)
FAM_ONEHOT = [c for c in dev.columns if c.startswith("fam__")]

BASE_OPTS = dict(exclude_postpeak=TRAIN_EXCLUDE_POSTPEAK, weighting=WEIGHTING, outlier_mode=FEATURE_OUTLIER_MODE, hall_factor=False)

def training_filter(tr, opts):
    keep = np.ones(len(tr), bool); info = {"train_rows_in": len(tr)}
    if opts["exclude_postpeak"]:
        keep &= ~tr["flag_postpeak_screen"].to_numpy(bool); info["removed_postpeak"] = int((~keep).sum())
    if opts["outlier_mode"] == "train_fold":
        base = tr[keep]; out = np.zeros(len(tr), bool)
        for c in OUTLIER_FEATURES:
            med = base[c].median(); mad = (base[c] - med).abs().median()
            if mad > 0:
                o = ((0.6745 * (tr[c] - med) / mad).abs() > ROBUST_Z).to_numpy(); info[f"outlier_{c}"] = int((keep & o).sum()); out |= o
        info["removed_outlier"] = int((keep & out).sum()); keep &= ~out
    t = tr[keep].copy()
    t["w"] = (1.0 / t.groupby("sample_key")["sample_key"].transform("size")) if opts["weighting"] == "equal_retained_sample" else t["w_pre"]
    info["train_rows_out"] = len(t)
    return t, info

def add_features(df, fam, glob, hall_factor=False):
    df = df.copy()
    df["mstar"], df["mstar_source"] = apply_masses(df, fam, glob)
    df["S_SPB_uVK"] = spb_seebeck_abs_uVK(df["n_m3"], df["T_K"], df["mstar"], hall_factor)
    df["s_SPB"] = df["S_SPB_uVK"] / KBE_UV
    if hall_factor:     # ln(n_true / N_c) with n_true = r_H(eta) n_H, i.e. (2/√π) F_1/2(eta)
        eta, lo, hi = spb_eta(df["n_m3"], df["T_K"], df["mstar"], True)
        with np.errstate(divide="ignore", invalid="ignore"):
            f12 = np.where(lo, np.sqrt(np.pi) / 2 * np.exp(eta), np.where(hi, 2 / 3 * eta ** 1.5, np.interp(eta, FT["eta"], FT["F12"])))
        df["ln_n_Nc"] = np.log(2 / np.sqrt(np.pi) * f12)
    else:
        df["ln_n_Nc"] = np.log(df["n_m3"] / N_c(df["T_K"], df["mstar"]))
    df["y"] = df["s"] - df["s_SPB"]
    return df

def build_cache(frame, seeds, opts, verbose=True):
    t0 = time.time(); cache = {}
    for sd in seeds:
        for k, (tr, te) in enumerate(splits_from(frame[f"fold_s{sd}"])):
            train, info = training_filter(frame.iloc[tr], opts)
            fam, glob = fit_mass_table(train, opts["hall_factor"])
            cache[(sd, k)] = dict(train=add_features(train, fam, glob, opts["hall_factor"]),
                                  test=add_features(frame.iloc[te], fam, glob, opts["hall_factor"]),
                                  te=te, fam=fam, glob=glob, info=info)
    if verbose: print(f"{len(cache)} fold frames built in {time.time() - t0:.0f} s ({opts})")
    return cache

def cv_predict(cache, fit_predict, frame=None):
    """fit_predict(train, test) -> predicted s. Returns {seed: out-of-fold array}."""
    n = len(frame if frame is not None else dev)
    out = {sd: np.full(n, np.nan) for sd in {k[0] for k in cache}}
    for (sd, k), c in cache.items():
        out[sd][c["te"]] = fit_predict(c["train"], c["test"])
    return out

def oof_column(cache, col, frame=None):
    n = len(frame if frame is not None else dev)
    out = {sd: np.full(n, np.nan, dtype=object if col == "mstar_source" else float) for sd in {k[0] for k in cache}}
    for (sd, k), c in cache.items(): out[sd][c["te"]] = c["test"][col].to_numpy()
    return out

CACHE = build_cache(dev, SEEDS, BASE_OPTS)
filt = pd.DataFrame([{"seed": sd, "fold": k, **c["info"]} for (sd, k), c in CACHE.items()])
display(filt.groupby("seed")[[c for c in filt.columns if c.startswith(("removed", "outlier_", "train_rows"))]].mean().round(1))

# A2 audit check: total training weight per retained sample under both policies (all development papers)
_t_new, _ = training_filter(dev, BASE_OPTS); _t_old, _ = training_filter(dev, {**BASE_OPTS, "weighting": "retained_fraction"})
wsum_new, wsum_old = _t_new.groupby("sample_key").w.sum(), _t_old.groupby("sample_key").w.sum()
REPORT["weights"] = {"policy": WEIGHTING, "samples_with_training_rows": len(wsum_new),
                     "current_policy_samples_weight_below_1": int((wsum_new < 1 - 1e-9).sum()),
                     "legacy_policy_samples_weight_below_1": int((wsum_old < 1 - 1e-9).sum()),
                     "legacy_policy_min_sample_weight": round(float(wsum_old.min()), 4)}
print("weights:", REPORT["weights"])

## 8. SPB baseline: how much does family calibration help?
Out-of-fold on development papers. The masses are fitted inside each training fold. The mass-stability table gives the number of fits behind every coefficient of variation, and flags masses that hit the search bound. (The earlier report's "most families vary by < 10 %" is regenerated here from the same definition: CV % = SD/mean over all seed × fold fits.)

In [ ]:
KEYS = sorted(CACHE)
OOF = {}                                   # (input_set, model) -> {seed: predicted s}
OOF[("—", "SPB m*=1")] = {sd: dev["S_SPB_abs_m1_uVK"].to_numpy() / KBE_UV for sd in SEEDS}
OOF[("—", "SPB global m*")] = cv_predict(CACHE, lambda tr, te: spb_seebeck_abs_uVK(te.n_m3, te.T_K, fit_mstar(tr.S_abs_uVK, tr.n_m3, tr.T_K, tr.w)) / KBE_UV)
OOF[("—", "SPB family m*")] = cv_predict(CACHE, lambda tr, te: te["s_SPB"].to_numpy())
MSTAR_OOF = oof_column(CACHE, "mstar"); MSTAR_SRC_OOF = oof_column(CACHE, "mstar_source")

def seed_table(keys, frame=None):
    frame = dev if frame is None else frame; rows = []
    for key in keys:
        for sd, p_ in OOF[key].items():
            rows.append({"input set": key[0], "model": key[1], "seed": sd, **score_s(frame["s"].to_numpy(), p_, frame)})
    return pd.DataFrame(rows)

def summary_table(per_seed, boot_keys=None):
    g = per_seed.groupby(["input set", "model"], sort=False)
    t = g[["R2", "MAE_uVK", "RMSE_uVK", "MAE_macro_sample_uVK", "MAE_macro_paper_uVK"]].mean()
    t["R2_split_SD"] = g["R2"].std(); t["MAE_split_SD"] = g["MAE_uVK"].std()
    if boot_keys:
        ci = {k: BOOT.ci(dev["s"].to_numpy(), OOF[k][PRIMARY]) for k in boot_keys}
        t["R2_CI95 (paper bootstrap, primary seed)"] = [ci.get(i, {}).get("R2_CI95") for i in t.index]
        t["MAE_CI95"] = [ci.get(i, {}).get("MAE_CI95") for i in t.index]
    return t

BOOT = PaperBootstrap(dev["SID"].to_numpy())
calib_keys = [("—", "SPB m*=1"), ("—", "SPB global m*"), ("—", "SPB family m*")]
calib_tab = summary_table(seed_table(calib_keys), calib_keys)
print("DOMAIN:", DOMAIN, "\nPoint-weighted scores unless labelled macro; ± = SD over fold seeds (split sensitivity):")
display(calib_tab.round(3)); save_table(calib_tab, "spb_calibration_dev_cv")

# mass stability across all seed × fold fits
mrows = [{"seed": sd, "fold": k, "family": f, "m_rel": v} for (sd, k), c in CACHE.items() for f, v in {**c["fam"], "__global__": c["glob"]}.items()]
mdf = pd.DataFrame(mrows)
stab = mdf.groupby("family")["m_rel"].agg(n_fits="size", mean="mean", sd="std", min="min", max="max")
stab["cv_%"] = 100 * stab["sd"] / stab["mean"]
stab["at_bound"] = (stab["max"] >= M_BOUNDS[1] * 0.999) | (stab["min"] <= M_BOUNDS[0] * 1.001)
stab = stab.sort_values("cv_%"); save_table(stab, "mass_stability_dev_folds")
print(f"family masses over {len(CACHE)} fits (n_fits < {len(CACHE)} = family below MIN_TRAIN_PAPERS in some folds):"); display(stab.round(3))
t_all, _ = training_filter(dev, BASE_OPTS)
FAM_DEVALL, GLOB_DEVALL = fit_mass_table(t_all)
fam_only = stab.drop(index="__global__")
REPORT["masses"] = {"families_with_mass_any_fold": len(fam_only),
                    "families_cv_below_10pct": int((fam_only["cv_%"] < 10).sum()),
                    "families_fewer_than_5_fits": int((fam_only["n_fits"] < 5).sum()),
                    "families_at_bound": fam_only.index[fam_only.at_bound].tolist(),
                    "cv_percent": fam_only["cv_%"].round(2).to_dict(), "n_fits": fam_only["n_fits"].to_dict(),
                    "devall_family_masses": {k: round(v, 4) for k, v in FAM_DEVALL.items()}, "devall_global_mass": round(GLOB_DEVALL, 4),
                    "fitted_masses_devall": len(FAM_DEVALL) + 1}

In [ ]:
# Pisarenko panels (250–350 K points, SPB at 300 K with the all-development family mass; descriptive)
fams = [f for f in t_all.groupby("family").SID.nunique().sort_values(ascending=False).index if f in FAM_DEVALL][:16]
nc = 4; nr = int(np.ceil(len(fams) / nc))
fig, ax = plt.subplots(nr, nc, figsize=(4.6 * nc, 3.6 * nr)); ax = np.atleast_1d(ax).ravel()
ng = np.logspace(17, 22.5, 200)
for a, f in zip(ax, fams):
    g_ = dev[(dev.family == f) & dev.T_K.between(250, 350)]
    a.scatter(g_.log10_n_cm3, g_.S_abs_uVK, s=7, alpha=.5, color=CAT[0])
    a.plot(np.log10(ng), spb_seebeck_abs_uVK(ng * 1e6, 300, FAM_DEVALL[f]), color=CAT[1], label=f"m_rel = {FAM_DEVALL[f]:.2f}")
    a.plot(np.log10(ng), spb_seebeck_abs_uVK(ng * 1e6, 300, 1.0), "--", color=INK2, lw=.9, label="m_rel = 1")
    a.set_title(f[:38], fontsize=10); a.set_ylim(0, 700); a.legend(fontsize=8.5, frameon=False)
for a in ax[len(fams):]: a.axis("off")
fig.supxlabel("log₁₀ n (cm⁻³)"); fig.supylabel("|S| (µV/K)")
fig.suptitle("Pisarenko view per family (development papers, descriptive)", x=0.01, ha="left")
plt.tight_layout(); save_fig(fig, "fig_pisarenko_families"); plt.show()

### 8.1 K-means composition clusters (exploratory, development papers only; C1)
Features: square-root element fractions (a Hellinger embedding) plus 0.3 × standardised mean electronegativity and valence. The **standardisation is fitted on each fold's training samples** and applied unchanged to the validation samples. K-means is refitted per fold, and one mass is fitted per cluster. The silhouette scan is descriptive only. Nothing downstream uses K-means.

In [ ]:
if RUN_KMEANS:
    ELEMENTS = sorted({e for c in dev.composition.unique() for e in (comp_cache.get(c) or {})})
    def comp_matrix(df, mu_sd):
        X = np.array([[np.sqrt((comp_cache.get(c) or {}).get(e, 0.0)) for e in ELEMENTS] for c in df["composition"]])
        ex = (df[["el_X_mean", "el_n_valence_mean"]].to_numpy() - mu_sd[0]) / mu_sd[1]
        return np.hstack([X, 0.3 * ex])
    def fit_scale(df):
        s_ = df.drop_duplicates("sample_key")[["el_X_mean", "el_n_valence_mean"]]
        return s_.mean().to_numpy(), s_.std().to_numpy()
    K = dev["group"].nunique()
    samp = dev.drop_duplicates("sample_key")
    Xs = comp_matrix(samp, fit_scale(dev))
    ks = [k_ for k_ in range(5, 61, 5) if k_ < len(samp)]
    sil = [silhouette_score(Xs, KMeans(n_clusters=k_, n_init=10, random_state=0).fit_predict(Xs)) for k_ in ks]
    km = KMeans(n_clusters=K, n_init=20, random_state=0).fit(Xs)
    ari = adjusted_rand_score(samp["group"], km.labels_)
    def kmeans_fp(tr, te):
        sc = fit_scale(tr)
        kmf = KMeans(n_clusters=K, n_init=10, random_state=0).fit(comp_matrix(tr.drop_duplicates("sample_key"), sc))
        tr = tr.assign(family=kmf.predict(comp_matrix(tr, sc)).astype(str)); te = te.assign(family=kmf.predict(comp_matrix(te, sc)).astype(str))
        fam, glob = fit_mass_table(tr); mm, _ = apply_masses(te, fam, glob)
        return spb_seebeck_abs_uVK(te.n_m3, te.T_K, mm) / KBE_UV
    prim = {k: v for k, v in CACHE.items() if k[0] == PRIMARY}
    OOF[("—", f"SPB K-means m* (K={K})")] = cv_predict(prim, kmeans_fp)
    y_ = dev["s"].to_numpy()
    km_tab = pd.DataFrame([{"model": k[1], **score_s(y_, OOF[k][PRIMARY], dev)} for k in calib_keys + [("—", f"SPB K-means m* (K={K})")]]).set_index("model")
    print(f"K = {K}, adjusted Rand index vs rule-based groups (development samples) = {ari:.3f}; primary seed:"); display(km_tab.round(2))
    save_table(km_tab, "kmeans_vs_rules_primary_seed")
    fig, ax = plt.subplots(figsize=(5.5, 3.2)); ax.plot(ks, sil, "o-", color=CAT[0]); ax.axvline(K, color=CAT[1], ls="--", label=f"K used = {K}")
    ax.set(xlabel="k", ylabel="silhouette", title="K-means silhouette (development samples, descriptive)"); ax.legend(frameon=False)
    plt.tight_layout(); save_fig(fig, "fig_kmeans_silhouette"); plt.show()

## 9. Input decision and the baseline ladder (development papers)
### 9.1 Does the correction need s_SPB as an *input*?
s_SPB stays the **baseline** in both versions. The only question is whether the correction model also takes it as an input. Same seeds, folds, rows and weights for both. s_SPB is dropped only if R² falls by less than `R2_TOLERANCE` **and** MAE rises by less than `MAE_TOLERANCE_UVK` (means over seeds). This decision is made on development papers only.

In [ ]:
def xgb_correction(inputs):
    def f(tr, te):
        return te["s_SPB"].to_numpy() + xgb.XGBRegressor(**XGB_PARAMS).fit(tr[inputs], tr["y"], sample_weight=tr["w"]).predict(te[inputs])
    return f
def linear_correction(inputs):
    def f(tr, te):
        return te["s_SPB"].to_numpy() + LinearRegression().fit(tr[inputs], tr["y"], sample_weight=tr["w"]).predict(te[inputs])
    return f
def quad_model(): return make_pipeline(StandardScaler(), PolynomialFeatures(2, include_bias=False), Ridge(alpha=QUAD_RIDGE_ALPHA))
def quadratic_correction(inputs):
    def f(tr, te):
        return te["s_SPB"].to_numpy() + quad_model().fit(tr[inputs], tr["y"], ridge__sample_weight=tr["w"]).predict(te[inputs])
    return f
def xgb_direct_calibrated(inputs):          # A3: predicts s directly; ln(n/N_c) still carries the fold-fitted family mass
    inp = [c for c in inputs if c != "s_SPB"]
    return lambda tr, te: xgb.XGBRegressor(**XGB_PARAMS).fit(tr[inp], tr["s"], sample_weight=tr["w"]).predict(te[inp])
def xgb_raw(set_name):                       # A3: raw n, T (+ same transport inputs) + family one-hot; no target-calibrated transform
    inp = RAW_INPUT_SETS[set_name] + FAM_ONEHOT
    return lambda tr, te: xgb.XGBRegressor(**XGB_PARAMS).fit(tr[inp], tr["s"], sample_weight=tr["w"]).predict(te[inp])
def xgb_dimensional(tr, te):                 # C4: legacy dimensional inputs, residual in µV/K, on the SAME rows/folds/weights
    r = xgb.XGBRegressor(**XGB_PARAMS).fit(tr[DIMENSIONAL_INPUTS], tr["S_abs_uVK"] - tr["S_SPB_uVK"], sample_weight=tr["w"])
    return (te["S_SPB_uVK"].to_numpy() + r.predict(te[DIMENSIONAL_INPUTS])) / KBE_UV

t0 = time.time()
for name, inp in CANDIDATE_SETS.items():
    OOF[("candidate", name)] = cv_predict(CACHE, xgb_correction(inp))
print(f"candidate sets done ({time.time() - t0:.0f} s)")
cand = seed_table([("candidate", n_) for n_ in CANDIDATE_SETS]).set_index(["model", "seed"])
decision = []
for with_, without in PAIRS:
    a, b = cand.loc[with_], cand.loc[without]
    dR2, dMAE = b.R2 - a.R2, b.MAE_uVK - a.MAE_uVK
    ok = (dR2.mean() > -R2_TOLERANCE) and (dMAE.mean() < MAE_TOLERANCE_UVK)
    decision.append({"input set": with_.split(" | ")[0], "R² with s_SPB input": a.R2.mean(), "R² without": b.R2.mean(),
                     "ΔR² (without − with)": dR2.mean(), "ΔR² split SD": dR2.std(), "ΔMAE µV/K": dMAE.mean(),
                     "seeds not worse": int((dR2 >= -1e-4).sum()), "decision": "drop s_SPB input" if ok else "keep s_SPB input",
                     "use": without if ok else with_})
decision = pd.DataFrame(decision).set_index("input set")
FINAL_SETS = {u.split(" | ")[0]: CANDIDATE_SETS[u] for u in decision["use"]}
display(decision.round(4)); save_table(decision, "input_decision")
print("FINAL input sets:", FINAL_SETS)

### 9.2 Baseline ladder
| Level | Model | Question it answers |
|---|---|---|
| 1 | SPB (family m*) | how far does calibrated physics alone get? |
| 2a / 2b | SPB + linear / + quadratic correction | does a low-order fix already do it? (the linear model already sees nonlinear SPB transforms of n and T) |
| 3 | SPB + PySR | *next stage, not in this notebook* |
| 4 | SPB + XGBoost correction | flexible correction with the same inputs |
| 4b | **Direct XGBoost on calibrated dimensionless features** | predicts s without the explicit baseline. Still physics-informed and family-calibrated through ln(n/N_c), so it isolates the value of the *explicit SPB baseline*, not of physics as a whole |
| 4c | **Raw-input XGBoost** | raw T, log₁₀ n, the same transport and composition inputs, and the family label (one-hot). No target-calibrated transform: the physics-free comparison |
| C4 | Dimensional SPB + XGBoost | legacy dimensional inputs, on **the same rows, folds, weights and filters** as the dimensionless version (paired) |

In [ ]:
t0 = time.time(); LADDER = {}
for set_name, inp in FINAL_SETS.items():
    models = [("1", "SPB (family m*)", None), ("2a", "SPB + linear", linear_correction(inp)),
              ("2b", "SPB + quadratic", quadratic_correction(inp)), ("4", "SPB + XGBoost", xgb_correction(inp)),
              ("4b", "Direct XGBoost (calibrated dimensionless features)", xgb_direct_calibrated(inp)),
              ("4c", "Raw-input XGBoost (no calibrated mass)", xgb_raw(set_name))]
    for level, name, fn in models:
        OOF[(set_name, name)] = OOF[("—", "SPB family m*")] if fn is None else cv_predict(CACHE, fn)
        LADDER[(set_name, name)] = level
    print(f"  {set_name} done ({time.time() - t0:.0f} s)")
OOF[("Hall + mob + val", "Dimensional SPB + XGBoost (legacy inputs, paired)")] = cv_predict(CACHE, xgb_dimensional)
LADDER[("Hall + mob + val", "Dimensional SPB + XGBoost (legacy inputs, paired)")] = "C4"
ladder_seed = seed_table(list(LADDER))
ladder_tab = summary_table(ladder_seed, list(LADDER))
ladder_tab.insert(0, "level", [LADDER[i] for i in ladder_tab.index])
print("DOMAIN:", DOMAIN)
print("Point-weighted R²/MAE/RMSE over pooled out-of-fold points (mean over seeds); macro = per-sample / per-paper MAE averaged;")
print("split SD = SD over fold seeds (split sensitivity, not a CI); CI95 = paper-level cluster bootstrap on the primary seed.")
display(ladder_tab.round(3)); save_table(ladder_tab, "baseline_ladder_dev"); ladder_seed.to_csv(OUT / "tables" / "baseline_ladder_per_seed.csv", index=False)

In [ ]:
# paired comparisons (primary seed, paper-level bootstrap on identical rows)
y_ = dev["s"].to_numpy(); P = PRIMARY; S2 = list(FINAL_SETS)
pairs_cmp = []
for st in S2:
    pairs_cmp += [(st, "SPB + quadratic", "SPB + XGBoost"), (st, "SPB + linear", "SPB + quadratic"),
                  (st, "SPB + XGBoost", "Direct XGBoost (calibrated dimensionless features)"),
                  (st, "Direct XGBoost (calibrated dimensionless features)", "Raw-input XGBoost (no calibrated mass)")]
pairs_cmp.append(("Hall + mob + val", "SPB + XGBoost", "Dimensional SPB + XGBoost (legacy inputs, paired)"))
pc = pd.DataFrame([{"input set": st, "a": a, "b": b, **BOOT.paired(y_, OOF[(st, a)][P], OOF[(st, b)][P])} for st, a, b in pairs_cmp])
print("Paired differences b − a (negative ΔMAE = b better); 95 % paper-bootstrap CI:"); display(pc.round(4)); save_table(pc, "paired_comparisons", index=False)

fig, ax = plt.subplots(1, len(FINAL_SETS), figsize=(7.4 * len(FINAL_SETS), 4.2), sharex=True, squeeze=False); ax = ax[0]
for a, st in zip(ax, FINAL_SETS):
    t = ladder_tab.loc[st]; t = t.loc[[i for i in t.index]]
    y = np.arange(len(t))[::-1]
    cols = [NEUTRAL if "SPB (family" in i else CAT[3] if "Raw" in i else CAT[4] if "Dimensional" in i else CAT[1] if "XGB" in i else CAT[0] for i in t.index]
    a.barh(y, t.R2, xerr=t.R2_split_SD, color=cols, height=0.62, error_kw=dict(ecolor=INK2, lw=1, capsize=2))
    for yi, v, e in zip(y, t.R2, t.R2_split_SD.fillna(0)): a.text(v + e + 0.015, yi, f"{v:.3f}", va="center", fontsize=9)
    a.set_yticks(y, [i.replace(" (", "\n(") for i in t.index], fontsize=9); a.set_xlim(0, 1)
    a.set_xlabel("point-weighted R², held-out development papers"); a.set_title(f"{st}: {', '.join(FINAL_SETS[st])}", loc="left", fontsize=10.5)
    a.xaxis.grid(True, color=GRID, lw=0.6); a.set_axisbelow(True)
fig.suptitle(f"Baseline ladder (mean ± split SD over {len(SEEDS)} seeds × {N_FOLDS} paper folds). Domain: {DOMAIN[:95]}…", x=0.01, ha="left", fontsize=10)
plt.tight_layout(); save_fig(fig, "fig_baseline_ladder"); plt.show()

In [ ]:
# parameter counts (one convention: INCLUDING the intercept) and all-development fits for the thesis text
PARAMS = {}; LINEAR_COEFS = {}; QUAD_TERMS = {}
dev_all = add_features(t_all, FAM_DEVALL, GLOB_DEVALL)
typical_masses = float(np.mean([len(c["fam"]) + 1 for c in CACHE.values()]))
for st, inp in FINAL_SETS.items():
    p_ = len(inp)
    lr = LinearRegression().fit(dev_all[inp], dev_all["y"], sample_weight=dev_all["w"])
    LINEAR_COEFS[st] = {"intercept": float(lr.intercept_), **{c: float(v) for c, v in zip(inp, lr.coef_)}}
    PARAMS[st] = {"inputs": p_, "SPB fitted masses (family + global), mean per fold": round(typical_masses, 1),
                  "SPB fitted masses, all-development fit": len(FAM_DEVALL) + 1,
                  "linear correction parameters incl. intercept": p_ + 1,
                  "quadratic correction parameters incl. intercept": (p_ + 1) * (p_ + 2) // 2,
                  "quadratic terms excl. intercept": (p_ + 1) * (p_ + 2) // 2 - 1,
                  "XGBoost trees": XGB_PARAMS["n_estimators"], "XGBoost max depth": XGB_PARAMS["max_depth"]}
REPORT["parameter_counts"] = PARAMS; REPORT["linear_coefficients_devall"] = LINEAR_COEFS
display(pd.DataFrame(PARAMS)); print("linear correction Δs = a + Σ b_i x_i (all development papers, descriptive):"); display(pd.DataFrame(LINEAR_COEFS).round(4))

## 10. Input importance for the SPB + XGBoost correction (development papers)
Three views, computed on every seed × fold. **Drop-column** (the main one): how much held-out MAE of Δs rises when the model is retrained without the input. It is robust to correlated inputs. **Permutation**: the rise when the input is shuffled in the validation papers. It underestimates correlated inputs. **SHAP**: mean |contribution| on validation papers. Units are Δs (multiply by 86.17 for µV/K).

In [ ]:
if RUN_IMPORTANCE:
    t0 = time.time(); IMP = {}
    for st, inp in FINAL_SETS.items():
        rows = []
        for (sd, k), c in CACHE.items():
            tr, te = c["train"], c["test"]
            full = xgb.XGBRegressor(**XGB_PARAMS).fit(tr[inp], tr["y"], sample_weight=tr["w"])
            base = mae(te["y"], full.predict(te[inp]))
            contrib = full.get_booster().predict(xgb.DMatrix(te[inp]), pred_contribs=True)[:, :-1]
            rng = np.random.default_rng(1000 * sd + k)
            for j, f in enumerate(inp):
                keep = [x for x in inp if x != f]
                d_drop = mae(te["y"], xgb.XGBRegressor(**XGB_PARAMS).fit(tr[keep], tr["y"], sample_weight=tr["w"]).predict(te[keep])) - base if keep else np.nan
                perm = []
                for _ in range(PERM_REPEATS):
                    Xp = te[inp].copy(); Xp[f] = rng.permutation(Xp[f].to_numpy()); perm.append(mae(te["y"], full.predict(Xp)) - base)
                rows.append({"seed": sd, "fold": k, "input": f, "drop_column": d_drop, "permutation": float(np.mean(perm)),
                             "shap": float(np.abs(contrib[:, j]).mean())})
        imp = pd.DataFrame(rows)
        agg = imp.groupby("input").agg(drop_column=("drop_column", "mean"), drop_column_sd=("drop_column", "std"),
                                       permutation=("permutation", "mean"), permutation_sd=("permutation", "std"),
                                       shap=("shap", "mean"), shap_sd=("shap", "std"))
        agg["share_of_fits_helpful"] = imp.assign(h=imp.drop_column > 0).groupby("input")["h"].mean()
        IMP[st] = agg.sort_values("drop_column", ascending=False)
        imp.to_csv(OUT / "tables" / f"importance_per_fit_{st.replace(' + ', '_').replace(' ', '_')}.csv", index=False)
        save_table(IMP[st], f"importance_{st.replace(' + ', '_').replace(' ', '_')}")
        print(f"{st} ({time.time() - t0:.0f} s):"); display(IMP[st].round(4))
    fig, axes = plt.subplots(len(IMP), 3, figsize=(14, 1.6 + 0.7 * sum(len(v) for v in IMP.values())), squeeze=False)
    for row, (st, a_) in zip(axes, IMP.items()):
        order = a_.index[::-1]; y = np.arange(len(order))
        for a, (col, sd_, ttl) in zip(row, [("drop_column", "drop_column_sd", "Drop-column (main)"), ("permutation", "permutation_sd", "Permutation"), ("shap", "shap_sd", "SHAP")]):
            a.barh(y, a_.loc[order, col], xerr=a_.loc[order, sd_], color=CAT[0], height=0.55, error_kw=dict(ecolor=INK2, lw=1, capsize=3))
            a.axvline(0, color=INK2, lw=.8); a.set_yticks(y, order); a.set_title(f"{st}: {ttl}", loc="left", fontsize=10)
            a.xaxis.grid(True, color=GRID, lw=.6); a.set_axisbelow(True)
    fig.suptitle("Input importance for Δs (mean ± SD over paper-held-out fits; development papers)", x=0.01, ha="left")
    plt.tight_layout(); save_fig(fig, "fig_input_importance"); plt.show()

## 11. Family transferability and pooling (exploratory)
**Question.** Do groups (families or fallback classes) deviate from SPB in the same way, so that they could share one correction model? And does a pooled model beat per-group models?

**Distance.** E(A→B) is the MAE on group B of a correction model trained on group A. E\*(B) is B's reference error: the better of its own paper-grouped CV model and "no correction". R(A→B) = E(A→B)/E\*(B), and the distance is ½[R(A→B)+R(B→A)] − 1, floored at 0. Groups are merged by average-linkage clustering at `MERGE_THRESHOLD`.

**Paper independence (A6):**
* **Self-reference.** For each inner split of B's papers, the masses are refitted on all training rows *except* the inner-validation papers. The inner-validation labels never touch the calibration.
* **A→B.** Masses are refitted without any of B's papers, and **every paper that also appears in B is removed from A's training rows**, so no paper is on both sides.

**Nested evaluation.** The tree is rebuilt inside each outer training set (primary seed) and scored on the outer validation papers. **Capacity (A7).** With `CAPACITY_MODE="equal"`, pooled, per-group and merged models all use `XGB_PARAMS`. Rows in groups too small for the tree are predicted by the pooled model (**fallback**), and their share is reported. The result is evidence about *these tested strategies*. It is not a causal claim that family models fail intrinsically.

In [ ]:
if RUN_TRANSFER:
    TIN = FINAL_SETS[TRANSFER_SET]
    GROUP_PARAMS = XGB_PARAMS if CAPACITY_MODE == "equal" else XGB_SMALL
    def fitx(df, params=None):
        return xgb.XGBRegressor(**(params or GROUP_PARAMS)).fit(df[TIN], df["y"], sample_weight=df["w"])
    def paper_folds(df, k, seed=0):
        sids = sorted(df.SID.unique(), key=lambda s_: sha(f"{seed}-{s_}")); fmap = {s_: i % k for i, s_ in enumerate(sids)}
        return df.SID.map(fmap).to_numpy()
    def eligible_groups(df):
        cnt = df.groupby("group").agg(papers=("SID", "nunique"), points=("point_id", "size"))
        return cnt[(cnt.papers >= MIN_GROUP_PAPERS) & (cnt.points >= MIN_GROUP_POINTS)].index.tolist()

    def transfer_matrix(df):
        """df: filtered, weighted TRAINING rows (raw columns). Masses are refitted inside every inner split."""
        G = eligible_groups(df); info = {}
        for b in G:
            sub = df[df.group == b]; pf = paper_folds(sub, min(SELF_FOLDS, sub.SID.nunique()))
            e_mod, e_zero = [], []
            for kk in np.unique(pf):
                hold = set(sub.SID[pf == kk])
                fam, glob = fit_mass_table(df[~df.SID.isin(hold)])
                trb, teb = add_features(sub[pf != kk], fam, glob), add_features(sub[pf == kk], fam, glob)
                e_mod += list(np.abs(teb.y - fitx(trb).predict(teb[TIN]))); e_zero += list(np.abs(teb.y))
            info[b] = {"papers": sub.SID.nunique(), "points": len(sub), "E_own_model": np.mean(e_mod), "E_no_correction": np.mean(e_zero)}
            info[b]["E_ref"] = min(info[b]["E_own_model"], info[b]["E_no_correction"])
        R = pd.DataFrame(np.nan, index=G, columns=G); shared = {}
        for b in G:
            sids_b = set(df.SID[df.group == b])
            fam, glob = fit_mass_table(df[~df.SID.isin(sids_b)])
            teb = add_features(df[df.group == b], fam, glob)
            for a in G:
                if a == b: R.loc[a, b] = 1.0; continue
                tra = df[(df.group == a) & ~df.SID.isin(sids_b)]
                shared[(a, b)] = int(df[df.group == a].SID.isin(sids_b).sum())
                if len(tra) < MIN_GROUP_POINTS or tra.SID.nunique() < 2: continue
                R.loc[a, b] = mae(teb.y, fitx(add_features(tra, fam, glob)).predict(teb[TIN])) / info[b]["E_ref"]
        Rv = R.values; sym = np.nanmean(np.stack([Rv, Rv.T]), axis=0)
        Dv = np.clip(np.nan_to_num(sym, nan=2.0) - 1, 0, None); np.fill_diagonal(Dv, 0)
        return pd.DataFrame(Dv, index=G, columns=G), R, pd.DataFrame(info).T, shared

    def merge_groups(D):
        Z = linkage(squareform(D.values, checks=False), method="average")
        return Z, dict(zip(D.index, fcluster(Z, t=MERGE_THRESHOLD, criterion="distance")))

    multi_group_papers = (t_all.groupby("SID")["group"].nunique() > 1)
    REPORT["transfer"] = {"papers_in_more_than_one_group_devall": int(multi_group_papers.sum()), "capacity_mode": CAPACITY_MODE}

    # ---- descriptive tree on all development papers (for the figure; not an evaluation)
    t0 = time.time()
    D_all, R_all, info_all, shared_all = transfer_matrix(t_all)
    Z_all, lab_all = merge_groups(D_all)
    print(f"descriptive tree: {len(D_all)} groups ({time.time() - t0:.0f} s); "
          f"{sum(v > 0 for v in shared_all.values())} ordered pairs had shared-paper rows removed from A's training data")
    members = pd.Series(lab_all).groupby(pd.Series(lab_all)).apply(lambda s: list(s.index))
    merge_table = pd.DataFrame({"group": list(lab_all), "cluster": list(lab_all.values())}).join(info_all, on="group").sort_values(["cluster", "points"], ascending=[True, False])
    display(merge_table.round(3)); save_table(merge_table, "merge_table_devall_descriptive", index=False)
    save_table(D_all, "transfer_distance_devall"); save_table(R_all, "transfer_ratio_devall")

    fig = plt.figure(figsize=(18, 0.42 * len(D_all) + 3))
    gs = fig.add_gridspec(1, 4, width_ratios=[0.7, 0.55, 1.35, 0.04], wspace=0.05)          # gs[1] = label space
    axd, axh, axc = fig.add_subplot(gs[0]), fig.add_subplot(gs[2]), fig.add_subplot(gs[3])
    dn = dendrogram(Z_all, orientation="left", no_labels=True, ax=axd, color_threshold=MERGE_THRESHOLD, above_threshold_color=NEUTRAL)
    axd.set_ylim(0, 10 * len(D_all)); axd.set_yticks([]); axd.spines["left"].set_visible(False)
    axd.axvline(MERGE_THRESHOLD, color=INK2, ls="--", lw=.9); axd.set_xlabel("transfer distance (0 = interchangeable)")
    order = [D_all.index[i] for i in dn["leaves"]]
    im = axh.imshow(np.minimum(D_all.loc[order, order].values, 1.0), cmap="Blues_r", vmin=0, vmax=1, origin="lower", aspect="auto",
                    extent=(-0.5, len(order) - 0.5, -0.5, len(order) - 0.5))
    axh.set_xticks(range(len(order)), [o.replace("[fallback] ", "▸ ")[:18] for o in order], rotation=90, fontsize=8.5)
    axh.set_yticks(range(len(order)), [f"{o.replace('[fallback] ', '▸ ')[:42]} ({int(info_all.loc[o, 'papers'])})" for o in order], fontsize=9)
    axh.tick_params(length=0)
    fig.colorbar(im, cax=axc, label="distance (capped at 1)")
    fig.suptitle("Residual-model transferability between groups (development papers; descriptive, exploratory). Label: group (papers)", x=0.01, ha="left")
    save_fig(fig, "fig_transfer_tree"); plt.show()

In [ ]:
if RUN_TRANSFER:
    t0 = time.time()
    strat = ["SPB (family m*) only", "SPB + per-group XGB", "SPB + merged-group XGB", "SPB + one pooled XGB"]
    NP = {s_: np.full(len(dev), np.nan) for s_ in strat}; fallback = np.zeros(len(dev), bool); comerge = []
    for k in range(N_FOLDS):
        c = CACHE[(PRIMARY, k)]; tr, te, idx = c["train"], c["test"], c["te"]
        D_k, _, _, _ = transfer_matrix(tr); _, lab_k = merge_groups(D_k); comerge.append(lab_k)
        p_pool = fitx(tr, XGB_PARAMS).predict(te[TIN]); p_fam, p_mrg = p_pool.copy(), p_pool.copy()
        fallback[idx] = ~te["group"].isin(D_k.index).to_numpy()
        for g_ in D_k.index:
            mte = (te.group == g_).to_numpy()
            if mte.any(): p_fam[mte] = fitx(tr[tr.group == g_]).predict(te[TIN][mte])
        for cl in set(lab_k.values()):
            gs_ = [g_ for g_ in lab_k if lab_k[g_] == cl]; mte = te.group.isin(gs_).to_numpy()
            if mte.any():
                prm = XGB_PARAMS if (CAPACITY_MODE == "equal" or len(gs_) > 1) else XGB_SMALL
                p_mrg[mte] = fitx(tr[tr.group.isin(gs_)], prm).predict(te[TIN][mte])
        base = te["s_SPB"].to_numpy()
        NP[strat[0]][idx] = base; NP[strat[1]][idx] = base + p_fam; NP[strat[2]][idx] = base + p_mrg; NP[strat[3]][idx] = base + p_pool
        print(f"outer fold {k}: {len(D_k)} groups -> {len(set(lab_k.values()))} merged ({time.time() - t0:.0f} s)")
    y_ = dev["s"].to_numpy()
    nest = pd.DataFrame({s_: {**score_s(y_, NP[s_], dev), **BOOT.ci(y_, NP[s_])} for s_ in strat}).T
    print(f"Nested, primary seed, capacity = {CAPACITY_MODE}; {fallback.mean():.1%} of validation points use the pooled fallback in "
          f"the per-group and merged strategies. DOMAIN: {DOMAIN}"); display(nest)
    by_grp = pd.DataFrame([{"group": g_, "papers": gg.SID.nunique(), "fallback": bool(fallback[gg.index].all()),
                            **{s_: mae(y_[gg.index], NP[s_][gg.index]) * KBE_UV for s_ in strat}}
                           for g_, gg in dev.groupby("group")]).set_index("group").sort_values("papers", ascending=False)
    display(by_grp.round(1))
    allg = sorted(set().union(*[set(c_) for c_ in comerge]))
    CM = pd.DataFrame(0.0, index=allg, columns=allg); CN = CM.copy()
    for lab_k in comerge:
        for a in lab_k:
            for b in lab_k: CN.loc[a, b] += 1; CM.loc[a, b] += lab_k[a] == lab_k[b]
    stab_m = CM / CN.replace(0, np.nan)
    save_table(nest, "nested_pooling_comparison"); save_table(by_grp, "nested_pooling_by_group"); save_table(stab_m, "comerge_stability")
    REPORT["transfer"].update({"fallback_share_validation_points": round(float(fallback.mean()), 4),
                               "nested": nest[["R2", "MAE_uVK", "MAE_macro_paper_uVK"]].round(4).to_dict(orient="index")})

## 12. Diagnostics and sensitivity analyses (development papers)
### 12.1 Residual structure: a descriptive decomposition, not a ceiling (A4)
For a residual $r = s - s_{\rm SPB}$ on a stated set of rows and masses, the table reports:
* **share explained by paper means** $= 1 - \sum_i (r_i - \bar r_{p(i)})^2 / \sum_i (r_i-\bar r)^2$, and the same for sample means;
* **"oracle" R²** $=R^2\big(s,\ s_{\rm SPB} + \bar r_{g(i)}\big)$, where $\bar r_g$ is the **true** mean residual of the row's own paper or sample.

The oracle estimates and evaluates each group offset on the same targets. It is not available at prediction time, and it is **not an upper bound**: a model can also capture temperature-dependent structure within a group, and inputs such as mobility, composition and n can correlate with sample-level differences. These numbers describe how the residual is distributed. They do not show that the offsets are unpredictable. The in-sample row uses masses fitted on all development papers. The out-of-fold rows use fold-fitted masses (primary seed).

In [ ]:
def decompose(r, s_true, base, frame, label):
    r = np.asarray(r, float); tot = np.sum((r - r.mean()) ** 2); out = {"residual": label, "rows": len(r)}
    for g_, name in (("SID", "paper"), ("sample_key", "sample")):
        gm = pd.Series(r).groupby(frame[g_].to_numpy()).transform("mean").to_numpy()
        out[f"share explained by {name} means"] = 1 - np.sum((r - gm) ** 2) / tot
        out[f"'oracle' R² with true {name} offsets"] = r2(s_true, base + gm)
    return out
dev_in = add_features(dev, FAM_DEVALL, GLOB_DEVALL)
best_key = ("Hall + mob + val", "SPB + XGBoost") if ("Hall + mob + val", "SPB + XGBoost") in OOF else list(LADDER)[3]
spb_oof = OOF[("—", "SPB family m*")][PRIMARY]; y_ = dev["s"].to_numpy()
decomp = pd.DataFrame([
    decompose(dev_in.y, y_, dev_in.s_SPB, dev, "in-sample: s − s_SPB, masses fitted on all development papers"),
    decompose(y_ - spb_oof, y_, spb_oof, dev, "out-of-fold: s − s_SPB (fold masses, primary seed)"),
    decompose(y_ - OOF[best_key][PRIMARY], y_, OOF[best_key][PRIMARY], dev, f"out-of-fold: s − ({best_key[1]}, {best_key[0]})")]).set_index("residual")
print("Development papers only. DOMAIN:", DOMAIN); display(decomp.round(4)); save_table(decomp, "residual_structure_descriptive")
REPORT["residual_structure"] = decomp.round(4).to_dict(orient="index")

### 12.2 Train-vs-validation fit of the actual models (replaces the unreproducible one-fold anecdote, C5)
For each fold of the primary seed: R² on the model's own training rows (in-sample) and on the held-out papers, for both the Δs target and the full s. The earlier report's one-fold `GradientBoostingRegressor` numbers came from code that isn't in the archive. They should be replaced by this table or removed.

In [ ]:
fit_rows = []
for st, inp in FINAL_SETS.items():
    for name, mk in [("SPB + quadratic", lambda tr, i=inp: quad_model().fit(tr[i], tr["y"], ridge__sample_weight=tr["w"])),
                     ("SPB + XGBoost", lambda tr, i=inp: xgb.XGBRegressor(**XGB_PARAMS).fit(tr[i], tr["y"], sample_weight=tr["w"]))]:
        for k in range(N_FOLDS):
            c = CACHE[(PRIMARY, k)]; tr, te = c["train"], c["test"]; mdl = mk(tr)
            ptr, pte = mdl.predict(tr[inp]), mdl.predict(te[inp])
            fit_rows.append({"input set": st, "model": name, "fold": k, "train R² (Δs)": r2(tr.y, ptr), "validation R² (Δs)": r2(te.y, pte),
                             "train R² (s)": r2(tr.s, tr.s_SPB + ptr), "validation R² (s)": r2(te.s, te.s_SPB + pte)})
fitdiag = pd.DataFrame(fit_rows); fitdiag_sum = fitdiag.groupby(["input set", "model"]).agg(["mean", "std"]).drop(columns="fold")
display(fitdiag_sum.round(3)); fitdiag.to_csv(OUT / "tables" / "fit_diagnostic_primary_seed.csv", index=False)

### 12.3 Sensitivity to training policies and the Hall factor (paired: same validation rows)
Each variant rebuilds every fold with **one** change. Only training rows, weights or masses differ, so the scores are paired with the main run.

| Variant | Change |
|---|---|
| post-peak filter off (B2) | post-peak screened points kept in training |
| legacy weighting (A2) | weights set before the training filters (retained-fraction weighting) |
| outlier filter off (A8) | no robust-z training filter |
| Hall factor r_H(η) (B6) | n = r_H(η)·n_H with the acoustic-SPB Hall factor, for masses, s_SPB and ln(n/N_c) |

In [ ]:
if RUN_SENSITIVITY:
    VARIANTS = {"main": BASE_OPTS,
                "post-peak filter off": {**BASE_OPTS, "exclude_postpeak": False},
                "legacy weighting (retained fraction)": {**BASE_OPTS, "weighting": "retained_fraction"},
                "outlier filter off": {**BASE_OPTS, "outlier_mode": "off"},
                "Hall factor r_H(eta)": {**BASE_OPTS, "hall_factor": True}}
    st = "Hall + mob + val" if "Hall + mob + val" in FINAL_SETS else list(FINAL_SETS)[0]; inp = FINAL_SETS[st]
    sens_rows, SENS_OOF = [], {}
    for vname, opts in VARIANTS.items():
        cache_v = CACHE if vname == "main" else build_cache(dev, SEEDS, opts)
        for mname, fn in [("SPB (family m*)", lambda tr, te: te["s_SPB"].to_numpy()), ("SPB + quadratic", quadratic_correction(inp)),
                          ("SPB + XGBoost", xgb_correction(inp))]:
            o = OOF[(st, mname)] if vname == "main" else cv_predict(cache_v, fn); SENS_OOF[(vname, mname)] = o
            for sd, p_ in o.items(): sens_rows.append({"variant": vname, "model": mname, "seed": sd, **score_s(y_, p_, dev)})
        if vname == "Hall factor r_H(eta)":
            fam_h, glob_h = fit_mass_table(training_filter(dev, opts)[0], True)
            mass_cmp = pd.DataFrame({"m_rel (r_H = 1)": pd.Series({**FAM_DEVALL, "__global__": GLOB_DEVALL}),
                                     "m_rel (r_H(eta))": pd.Series({**fam_h, "__global__": glob_h})})
            mass_cmp["ratio"] = mass_cmp.iloc[:, 1] / mass_cmp.iloc[:, 0]
    sens = pd.DataFrame(sens_rows)
    sens_tab = sens.groupby(["variant", "model"], sort=False)[["R2", "MAE_uVK", "MAE_macro_sample_uVK", "MAE_macro_paper_uVK"]].mean()
    sens_tab["R2_split_SD"] = sens.groupby(["variant", "model"], sort=False)["R2"].std()
    sens_tab["ΔMAE vs main (paired bootstrap, primary seed)"] = [
        None if v == "main" else BOOT.paired(y_, SENS_OOF[("main", m_)][PRIMARY], SENS_OOF[(v, m_)][PRIMARY])["dMAE_CI95"] for v, m_ in sens_tab.index]
    print(f"input set: {st}. DOMAIN: {DOMAIN}"); display(sens_tab.round(3)); save_table(sens_tab, "sensitivity_training_policies")
    print("Family masses, all development papers, with and without the acoustic-SPB Hall factor (effective quantities in both cases):")
    display(mass_cmp.round(3)); save_table(mass_cmp, "hall_factor_mass_sensitivity")
    REPORT["sensitivity"] = sens_tab.drop(columns=["ΔMAE vs main (paired bootstrap, primary seed)"]).round(4).reset_index().to_dict(orient="records")

### 12.4 S–n pairing and data-provenance subsets (B4), and the temperature trend
Main out-of-fold predictions (primary seed) scored on subsets. These are **not paired** comparisons: the rows differ between subsets. A narrow pairing tolerance, the n source and reused n points are checked this way without rebuilding the dataset. To rebuild with a narrower tolerance, set `TOL_K` and rerun. The temperature table is descriptive. A positive mean correction at high T is an empirical trend, and this regression **does not identify its mechanism**. Post-peak bipolar behaviour would usually *reduce* |S|, and those points are removed from training anyway.

In [ ]:
sub_rows = []
masks = {"all": np.ones(len(dev), bool), "|ΔT| ≤ 2 K": dev.dT_K.le(2).to_numpy(), "|ΔT| ≤ 1 K": dev.dT_K.le(1).to_numpy(),
         "n reported": dev.n_source.eq("reported_n").to_numpy(), "n Hall-derived": dev.n_source.ne("reported_n").to_numpy(),
         "n point not reused": dev.n_point_reuse.eq(1).to_numpy(), "post-peak screened points": dev.flag_postpeak_screen.to_numpy(bool),
         "family label ambiguous": dev.family_ambiguous.to_numpy(bool)}
for nm, mk in masks.items():
    if mk.sum() < 20: continue
    for key in [("—", "SPB family m*"), best_key]:
        sub_rows.append({"subset": nm, "rows": int(mk.sum()), "papers": int(dev.SID[mk].nunique()), "model": key[1],
                         "R2": r2(y_[mk], OOF[key][PRIMARY][mk]), "MAE_uVK": mae(y_[mk], OOF[key][PRIMARY][mk]) * KBE_UV})
subs = pd.DataFrame(sub_rows); display(subs.round(3)); save_table(subs, "subset_scores_primary_seed", index=False)
tb = pd.cut(dev.T_K, [100, 200, 300, 400, 500, 600, 800, 1300])
trend = pd.DataFrame({"T bin (K)": tb, "r": y_ - spb_oof}).groupby("T bin (K)", observed=True)["r"].agg(["mean", "size"])
trend.columns = ["mean OOF SPB residual Δs", "rows"]; trend["mean residual (µV/K)"] = trend.iloc[:, 0] * KBE_UV
display(trend.round(3)); save_table(trend, "residual_vs_temperature_descriptive")

## 13. Freeze: dataset, configuration, predictions, models, report numbers, README and manifest
| File | Content |
|---|---|
| `data/frozen_dataset.csv` | every row of the final domain with `split` (`dev` / `reserved_after_exploration`), the development fold columns, the raw columns needed to recompute everything, and `*_devall` columns (masses fitted on **all** development papers). **Don't use `*_devall` for development CV.** CV must refit masses inside folds, as this notebook does. They are only for a final fit that is scored once on the reserved papers |
| `frozen_config.json` | every setting, environment and package versions, Starrydata release tag, URLs and raw hashes, reserved SIDs, final inputs, baseline scores |
| `data/oof_predictions.csv.gz` | per point × seed × model: point_id, SID, sample_key, seed, fold, input set, model, true and predicted s, fold mass and its source (family / global) |
| `models/` | all-development fits: family masses, linear and quadratic coefficients, XGBoost correction models (`.json`) |
| `report_numbers.json` | every number the written report quotes, regenerated from this run |
| `README.md`, `MANIFEST.json` | version map (which old notebook each part replaces), rules for later stages, SHA-256 of every output |

In [ ]:
# ---- frozen dataset
fz = pd.concat([dev.assign(split="dev"), RESERVED._df.assign(split="reserved_after_exploration")], ignore_index=True)
fz = add_features(fz.assign(s=fz.S_abs_uVK / KBE_UV, tau=fz.T_K / T_REF, val_mean=fz.el_n_valence_mean), FAM_DEVALL, GLOB_DEVALL) \
       .rename(columns={"mstar": "mstar_devall", "mstar_source": "mstar_source_devall", "s_SPB": "s_SPB_devall",
                        "ln_n_Nc": "ln_n_Nc_devall", "y": "y_devall", "S_SPB_uVK": "S_SPB_uVK_devall"})
keep = ["point_id", "split", "SID", "DOI", "sample_key", "sample_name", "composition", "host", "family", "family_candidates",
        "family_ambiguous", "broad_class", "class_level", "group", "carrier_type", "carrier_type_confidence", "n_source", "dT_K",
        "n_point_reuse", "S_curve_uid", "T_K", "n_m3", "log10_n_cm3", "log10_sigma_Sm", "el_n_valence_mean", "dopant_total_frac",
        "el_X_mean", "S_uVK", "S_abs_uVK", "flag_postpeak_screen", "flag_seebeck_sign_mixed", "s", "tau", "log10_mu", "val_mean",
        "mstar_devall", "mstar_source_devall", "s_SPB_devall", "ln_n_Nc_devall", "y_devall"] + fcols
fz = fz[[c for c in keep if c in fz.columns]]
fz.to_csv(OUT / "data" / "frozen_dataset.csv", index=False)
FROZEN_SHA = sha256_file(OUT / "data" / "frozen_dataset.csv")

# ---- out-of-fold predictions with mass provenance
fold_of = {sd: dev[f"fold_s{sd}"].to_numpy() for sd in SEEDS}
parts = []
for (st, mdl), per_seed in OOF.items():
    if st == "candidate": continue
    for sd, p_ in per_seed.items():
        ok = ~np.isnan(p_)
        parts.append(pd.DataFrame({"point_id": dev.point_id.to_numpy()[ok], "SID": dev.SID.to_numpy()[ok],
                                   "sample_key": dev.sample_key.to_numpy()[ok], "seed": sd, "fold": fold_of[sd][ok],
                                   "input_set": st, "model": mdl, "s_true": dev.s.to_numpy()[ok], "s_pred": p_[ok],
                                   "mstar_fold": MSTAR_OOF[sd][ok], "mstar_source": MSTAR_SRC_OOF[sd][ok], "weighting": WEIGHTING}))
if RUN_TRANSFER:
    for s_, p_ in NP.items():
        parts.append(pd.DataFrame({"point_id": dev.point_id, "SID": dev.SID, "sample_key": dev.sample_key, "seed": PRIMARY,
                                   "fold": fold_of[PRIMARY], "input_set": TRANSFER_SET, "model": f"nested: {s_}", "s_true": dev.s,
                                   "s_pred": p_, "mstar_fold": MSTAR_OOF[PRIMARY], "mstar_source": MSTAR_SRC_OOF[PRIMARY], "weighting": WEIGHTING}))
oof_long = pd.concat(parts, ignore_index=True)
oof_long.to_csv(OUT / "data" / "oof_predictions.csv.gz", index=False)
print("oof_predictions:", len(oof_long), "rows")

# ---- all-development model fits
(OUT / "models" / "masses_devall.json").write_text(json.dumps({"family": FAM_DEVALL, "global": GLOB_DEVALL,
    "rule": f">= {MIN_TRAIN_PAPERS} papers, bounds {M_BOUNDS}, loss weighted mean ||S| - S_SPB|, weighting {WEIGHTING}"}, indent=2))
MODEL_FILES = {}
for st, inp in FINAL_SETS.items():
    tag = st.replace(" + ", "_").replace(" ", "_")
    q = quad_model().fit(dev_all[inp], dev_all["y"], ridge__sample_weight=dev_all["w"])
    sc, pf, rg = q.named_steps["standardscaler"], q.named_steps["polynomialfeatures"], q.named_steps["ridge"]
    (OUT / "models" / f"linear_{tag}.json").write_text(json.dumps({"inputs": inp, **LINEAR_COEFS[st]}, indent=2))
    (OUT / "models" / f"quadratic_{tag}.json").write_text(json.dumps({"inputs": inp, "scaler_mean": sc.mean_.tolist(),
        "scaler_scale": sc.scale_.tolist(), "terms": pf.get_feature_names_out(inp).tolist(), "coef": rg.coef_.tolist(),
        "intercept": float(rg.intercept_), "ridge_alpha": QUAD_RIDGE_ALPHA}, indent=2))
    xm = xgb.XGBRegressor(**XGB_PARAMS).fit(dev_all[inp], dev_all["y"], sample_weight=dev_all["w"])
    xm.get_booster().save_model(str(OUT / "models" / f"xgb_correction_{tag}.json"))
    MODEL_FILES[st] = [f"linear_{tag}.json", f"quadratic_{tag}.json", f"xgb_correction_{tag}.json"]

# ---- configuration
ladder_json = {f"{i[0]} | {i[1]}": {k: (list(v) if isinstance(v, tuple) else v) for k, v in r.items()}
               for i, r in ladder_tab.round(4).iterrows()}
CONFIG = {"created_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
          "status": "development CV on papers reserved after exploratory analysis; not an independent external validation",
          "settings": SETTINGS, "environment": ENV, "provenance": PROVENANCE, "domain": DOMAIN,
          "frozen_dataset": {"file": "data/frozen_dataset.csv", "sha256": FROZEN_SHA, "rows": len(fz),
                             "dev_rows": int((fz.split == "dev").sum()), "reserved_rows": int((fz.split != "dev").sum()),
                             "dev_papers": int(dev.SID.nunique()), "reserved_papers": len(reserved_sids)},
          "reserved_SIDs": reserved_sids, "input_decision": decision.reset_index().to_dict(orient="records"),
          "final_input_sets": FINAL_SETS, "baselines_dev_cv": ladder_json,
          "training_rules": {"postpeak_screen": "removed from TRAINING only" if TRAIN_EXCLUDE_POSTPEAK else "kept",
                             "feature_outliers": FEATURE_OUTLIER_MODE, "weights": WEIGHTING, "m_star": "refit inside every training fold"},
          "rules_for_later_stages": ["read data/frozen_dataset.csv and frozen_config.json; do not re-derive splits or filters",
                                     "tune only on split == 'dev' with the fold_s* columns, refitting m* inside each training fold",
                                     "report the domain next to every score; label point-weighted metrics and add macro sample/paper MAE",
                                     "evaluate the reserved papers once, at the end, and describe them as reserved after exploratory analysis",
                                     "an independent validation needs new papers"],
          "model_files": MODEL_FILES}
(OUT / "frozen_config.json").write_text(json.dumps(CONFIG, indent=2, default=str))

In [ ]:
# ---- report numbers (audit §5: regenerated from this run, with populations named)
REPORT["dimensionless_ranges_common_subset"] = (dev_in[["s", "s_SPB", "y", "ln_n_Nc", "tau", "log10_mu", "val_mean", "mstar"]]
    .describe(percentiles=[.01, .5, .99]).T.round(4).to_dict(orient="index"))
REPORT["dimensionless_ranges_note"] = f"all variables on the same {len(dev_in)} development rows, masses fitted on all development papers"
REPORT["baseline_ladder_dev"] = ladder_json
REPORT["paired_comparisons_primary_seed"] = pc.round(4).astype({"dMAE_CI95": str, "dR2_CI95": str}).to_dict(orient="records")
REPORT["notes"] = {"carrier_type": "labels from Hall sign only; 'unknown' = no Hall curve. Quote the population with every share.",
                   "rounding": "round directly from the exported values, never twice",
                   "reserved": "papers reserved after exploratory analysis; their exposure in earlier notebooks is not quantified"}
(OUT / "report_numbers.json").write_text(json.dumps(REPORT, indent=2, default=str))

README = f"""# Seebeck SPB + correction pipeline: output package

Generated {CONFIG['created_utc']} by `Seebeck_SPB_pipeline.ipynb` (single notebook).
Status: {CONFIG['status']}.

## Domain
{DOMAIN}

## Data provenance
* Input mode: {INPUT_MODE}
* Starrydata release: {PROVENANCE.get('release', {}).get('tag', 'n/a')}  ({PROVENANCE.get('release', {}).get('base_url', '')})
* Raw file SHA-256: {json.dumps({k: v['sha256'] for k, v in PROVENANCE.get('raw_files', {}).items()})}
* Frozen dataset SHA-256: {FROZEN_SHA}

## Environment
{json.dumps(ENV)}

## Version map: this notebook replaces
| Old notebook | Role | Now |
|---|---|---|
| seebeck_dataset_starrydata_1 (1) | S–n extraction | Section 3 (Hall labels, B8 units, per-curve post-peak, strict interpolation) |
| seebeck_features_preprocessing (1) | older preprocessing (historical) | superseded |
| seebeck_features_preprocessing_1 (1) / _1 | updated preprocessing (identical code) | Section 4 |
| mstar_calibration (1), _2 (1), _3 (1) | earlier calibration stages (historical) | superseded |
| mstar_calibration_5 (1) / (2) | families + calibration (identical code) | Sections 5, 7, 8 |
| xgb_ranking_merging (1) | earlier ranking/merging (fold-averaged metrics; historical) | superseded |
| xgb_ranking_merging_2 (1) | input ranking, transfer tree | Sections 10, 11 |
| dimensionless_xgb (1) | dimensionless comparison | Section 9 (paired, C4) |
| freeze_setup (1) | reservation, ladder, freeze | Sections 6, 9, 13 |
The old output archives (`mstar_calibration*.zip`, `xgb_ranking_merging*.zip`, ...) are historical runs and aren't interchangeable.

## Rules for later stages (PySR, neural networks)
""" + "\n".join(f"* {r}" for r in CONFIG["rules_for_later_stages"]) + """

## Files
See MANIFEST.json (SHA-256 of every file).
"""
(OUT / "README.md").write_text(README)
manifest = {str(p.relative_to(OUT)): {"bytes": p.stat().st_size, "sha256": sha256_file(p)}
            for p in sorted(OUT.rglob("*")) if p.is_file() and p.name not in ("MANIFEST.json", "fermi_table_v2.npz")}
(OUT / "MANIFEST.json").write_text(json.dumps(manifest, indent=1))
shutil.make_archive(OUT_DIR, "zip", OUT)
print(f"{len(manifest)} files written to {OUT}/ and {OUT_DIR}.zip  (total runtime {(time.time() - T_START) / 60:.1f} min)")
if USE_DRIVE and IN_COLAB:
    shutil.copy(f"{OUT_DIR}.zip", Path(DRIVE_DIR) / f"{OUT_DIR}_{dt.datetime.now():%Y%m%d_%H%M}.zip"); print("copied to Drive")
if IN_COLAB:
    from google.colab import files as _files; _files.download(f"{OUT_DIR}.zip")

## 14. One-time evaluation on the reserved papers (locked)
Run this **once**, after the development protocol (inputs, models, PySR and neural-network choices) is final. It fits the masses and models on all development papers, predicts the reserved papers, and writes `reserved_evaluation.json` with a timestamp. Report the result as *evaluation on papers reserved after exploratory analysis*, with the domain, and don't tune anything afterwards. The size of any optimism from the earlier exposure is unknown.

To run it: set `EVALUATE_RESERVED = True` and `RESERVED_CONFIRM = CONFIRM_PHRASE` in Section 1.

In [ ]:
if EVALUATE_RESERVED:
    logf = OUT / "reserved_evaluation.json"
    if logf.exists(): raise RuntimeError(f"{logf} already exists: the reserved papers have been evaluated before.")
    res = RESERVED.open(RESERVED_CONFIRM)
    res = prepare(res.assign(**{f"fold_s{sd}": -1 for sd in SEEDS}))
    res = add_features(res, FAM_DEVALL, GLOB_DEVALL)
    out = {}
    for st, inp in FINAL_SETS.items():
        preds = {"SPB (family m*)": res["s_SPB"].to_numpy(),
                 "SPB + linear": res["s_SPB"].to_numpy() + LinearRegression().fit(dev_all[inp], dev_all.y, sample_weight=dev_all.w).predict(res[inp]),
                 "SPB + quadratic": res["s_SPB"].to_numpy() + quad_model().fit(dev_all[inp], dev_all.y, ridge__sample_weight=dev_all.w).predict(res[inp]),
                 "SPB + XGBoost": res["s_SPB"].to_numpy() + xgb.XGBRegressor(**XGB_PARAMS).fit(dev_all[inp], dev_all.y, sample_weight=dev_all.w).predict(res[inp])}
        rb = PaperBootstrap(res.SID.to_numpy())
        out[st] = {k: {**score_s(res.s.to_numpy(), p_, res), **rb.ci(res.s.to_numpy(), p_)} for k, p_ in preds.items()}
    logf.write_text(json.dumps({"evaluated_utc": dt.datetime.now(dt.timezone.utc).isoformat(), "domain": DOMAIN,
                                "status": "papers reserved after exploratory analysis", "frozen_dataset_sha256": FROZEN_SHA,
                                "scores": out}, indent=2, default=str))
    print(json.dumps(out, indent=1, default=str))
else:
    print("Reserved papers remain sealed:", RESERVED)